# RSNA Knee hidden-test-safe inference candidate

Generated from the verified public-score V52 notebook, keeping its packaged DINOv2
inference path and adding the public, hash-pinned RadImageNet E10 arm.  Configuration:
`per_target_nested`.  The scored run reads every hidden test DICOM; no visible-test prediction
or UID is embedded in this notebook.


In [8]:
from __future__ import annotations

import os

for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ.setdefault(_v, "4")

import gc
import hashlib
import json
import re
import time
import traceback
import threading
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import pandas as pd
import pydicom
import torch
import torch.nn as nn
import torch.nn.functional as F

# Use every accelerator that can execute the current PyTorch kernels.  Kaggle can
# attach a legacy GPU that torch.cuda.is_available() reports as usable even though
# the image has no compatible convolution kernel.  A real model-shaped probe makes
# that allocation fail closed to CPU before any ensemble member is consumed.
def _cuda_execution_probe(index):
    dev = torch.device(f"cuda:{index}")
    try:
        major, minor = torch.cuda.get_device_capability(index)
        probe = nn.Conv2d(3, 4, kernel_size=3, padding=1).eval().to(dev)
        with torch.inference_mode():
            out = probe(torch.zeros((1, 3, 16, 16), device=dev))
            if tuple(out.shape) != (1, 4, 16, 16):
                raise RuntimeError(f"unexpected CUDA probe shape {tuple(out.shape)}")
        torch.cuda.synchronize(index)
        print(f"cuda:{index} probe PASS (compute {major}.{minor})")
        del probe, out
        torch.cuda.empty_cache()
        return True
    except Exception as exc:
        print(f"cuda:{index} probe FAIL ({type(exc).__name__}: {exc}); using CPU fallback")
        try:
            torch.cuda.empty_cache()
        except Exception:
            pass
        return False

DEVS = []
if torch.cuda.is_available():
    DEVS = [torch.device(f"cuda:{i}") for i in range(torch.cuda.device_count())
            if _cuda_execution_probe(i)]
if not DEVS:
    DEVS = [torch.device("cpu")]
print(f"devices: {[str(d) for d in DEVS]}")


# The label extractor is defined in the cells above when this runs as a notebook. As a
# plain script it is imported from the package source, so the two paths share one
# definition rather than keeping a copy each.

T0 = time.time()
SEED = 2026
np.random.seed(SEED)
torch.manual_seed(SEED)

TARGETS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA",
           "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker's",
           "Contusion", "Fracture"]


# The centre crop has to be smaller than the smallest field of view in the corpus or it
# silently does nothing. Measured over every training series, the acquired field of view
# (Rows x PixelSpacing) has median 160 mm and runs from 70 to 320: a 160 mm crop is
# larger than the image in 60% of series and is skipped for all of them, which leaves
# their physical scale unnormalised. 130 mm is below the field of view of 99.6% of
# series and still contains the joint.
CROP_MM = 130.0

# Cache resolution. Everything downstream may downsample from this, so it is set by the
# most demanding configuration rather than by the default one.
CACHE_IMG = 336
GROUP = 3                  # slices per encoder input, stacked as the three channels
N_GROUP_MAX = 1
CACHE_FRACTION = 0.45      # share of free memory the pixel cache may take
CACHE_BUDGET_MAX_GB = 24.0 # hard ceiling regardless of what the machine reports
CACHE_BUDGET_GB = 12.0     # only the fallback, for a machine with no /proc/meminfo
TEST_SHARE = 0.30          # floor on the test corpus relative to the training one, since
                           # the visible test split is a stub and the scored one is not
HDR_THREADS = 16
PIX_THREADS = 12
ORDER_THREADS = 32         # slice-ordering is latency-bound on the mount, not CPU-bound
# Ceiling for the ordering pass. It has to be a ceiling because the pass is hundreds of
# thousands of small reads over a network mount, so its duration is a property of the
# mount rather than of the work, and varies between runs that do the same reading. It must not be a tight one: giving up leaves
# those series in file order, which is uncorrelated with anatomy, and that degradation is
# silent. So the ceiling sits well above what the pass ordinarily needs: its purpose is
# to stop the pass consuming the whole run on a slow mount, not to trim the ordinary
# case, and a ceiling tight enough to bind on a normal day would trade a silent
# degradation for a saving the run does not need.
ORDER_BUDGET_S = 5400

# Resolution is the axis under test. A feature of width d mm survives resampling only if
# the pixel pitch is at most d/2, and the pitch here is set by the crop above rather than
# by the acquired field of view: CROP_MM / P. At 224 px that is 0.58 mm, above the 0.5 mm
# a 1 mm tear needs; at 336 px it is 0.39 mm and clears it. Both configurations read the
# same cache, so the comparison isolates the resize.
RUNS = [
    {"name": "r224", "img": 224},
    {"name": "r336", "img": 336},
]

EPOCHS = 10
BATCH_STUDIES = 8          # a study is a bag of up to N_SLOT slot images
AUG_ROT_DEG = 8.0          # rigid jitter; see augment() for why neither flip is used
AUG_SCALE = 0.08
AUG_SHIFT = 0.05
AUG_INTENSITY = 0.10
LAT_MIN_OFFSET_MM = 20.0   # inside this the side is not readable from geometry; see
                           # side_from_geometry()
SLICE_BAND = (0.20, 0.80)  # fraction of the ordered stack read_slot samples across

# --- What a slice IS, as opposed to how many of them there are --------------- #
#
# A member is a function of the pixels it was fitted on, and img/crop_mm/slices/band do
# not determine those pixels by themselves. Four further decisions do, none of them
# visible in any shape:
#
#   order          which slice is the next one along the stack
#   lat            which knees are mirrored, and on what evidence
#   slot_fallback  whether a T1 slot may be filled from a series that is not T1
#   decode_fill    what stands in for a slice that would not decode
#
# `native` is the reading derived in the sections below. `legacy` is the reading an
# imported member was fitted under. A member read under the wrong one loads with every
# shape matching, runs, and writes a plausible submission computed from the wrong image -
# so the choice travels with the member and is part of the key that decides which members
# can share a decode. The legacy rules are reproduced rather than corrected: correcting
# them would hand that member pixels its weights never saw.
RULES_NATIVE = {"order": "normal", "lat": "centre",
                "slot_fallback": False, "decode_fill": "nearest"}
RULES_LEGACY = {"order": "dominant_axis", "lat": "corner_x",
                "slot_fallback": True, "decode_fill": "zero"}
RULES = dict(RULES_NATIVE)
LEGACY_LAT_OFFSET_MM = 5.0   # the dead zone the legacy laterality rule was fitted with

LR_HEAD = 1e-3
LR_BACKBONE = 8e-6         # the encoder is adapted, not retrained
UNFREEZE_LAST = 6          # trainable transformer blocks, from the output end
WEIGHT_DECAY = 0.02
EVAL_BATCH = 8
TIME_BUDGET = 8.0 * 3600

# Six slots: three planes crossed with the acquisition axes. The fat-suppressed
# fluid-sensitive series exist for nearly every study; the T1 and the non-suppressed
# fluid-sensitive series are scarcer, which is what the presence mask is for.
SLOTS_RECOVERED = [
    ("SAG_FLUID_FS", "Sagittal", True, True),
    ("COR_FLUID_FS", "Coronal", True, True),
    ("AX_FLUID_FS", "Axial", True, True),
    ("SAG_FLUID_NOFS", "Sagittal", True, False),
    ("COR_T1", "Coronal", False, False),
    ("SAG_T1", "Sagittal", False, False),
]

# The alternative: plane x the single axis the delivered flags carry, ignoring the
# recovered weighting. Kept as
# a switch so the choice of slot definition can be varied while everything else is held
# fixed. Under this scheme a `Struct` slot mixes T1 series with non-fat-suppressed PD/T2
# series, which carry very different tissue contrast.
SLOTS_PUBLIC = [
    ("SAG_FLUID", "Sagittal", None, True),
    ("COR_FLUID", "Coronal", None, True),
    ("AX_FLUID", "Axial", None, True),
    ("SAG_STRUCT", "Sagittal", None, False),
    ("COR_STRUCT", "Coronal", None, False),
    ("AX_STRUCT", "Axial", None, False),
]

SLOT_SCHEME = os.environ.get("SLOT_SCHEME", "recovered")
SLOTS = SLOTS_PUBLIC if SLOT_SCHEME == "public" else SLOTS_RECOVERED
N_SLOT = len(SLOTS)

# How many 384-wide parts the per-slot feature is built from. The encoder emits one
# vector per token; a slot feature is a fixed summary of that grid, and the summary an
# imported member was fitted with carries a third part.
POOL_PARTS = {"cls_mean": 2, "cls_mean_focal": 3}

# Which slots an imported member's attention is tilted toward, per diagnosis. Indices are
# into SLOTS. This is a fixed table rather than a learned parameter, so it is part of that
# member's definition and has to be reproduced exactly for its weights to mean anything.
SLOT_PRIOR_TABLE = {
    "ACL": (0, 3, 5), "MCL": (1, 4),
    "Medial Meniscus": (0, 1, 3, 4), "Lateral Meniscus": (0, 1, 3, 4),
    "Medial OA": (1, 4, 5), "Lateral OA": (1, 4, 5),
    "PF OA": (0, 2, 5), "Effusion": (0, 2), "Synovitis": (0, 2),
    "Baker's": (0,), "Contusion": (0, 1, 2), "Fracture": (0, 1, 2, 4, 5),
}
SLOT_PRIOR_STRENGTH = 0.55

FATSAT_OPTS = {"FS", "FATSAT", "FAT_SAT", "FSAT"}
_SEP = re.compile(r"[_\-.]")
_FATSAT_RX = re.compile(r"\bfs\b|fatsat|fat sat|\bstir\b|\bspair\b|\bspir\b|\bwe\b|"
                        r"water excit|\btirm\b|\bsting\b|\bfatsup\b")
_T1_RX = re.compile(r"\bt1\b|\bt1w\b")
_T2_RX = re.compile(r"\bt2\b|\bt2w\b")
_PD_RX = re.compile(r"\bpd\b|\bpdw\b|proton|\bdp\b|dens")


In [9]:
def log(msg):
    print(f"[{time.time() - T0:7.1f}s] {msg}", flush=True)


def find_root():
    for c in [Path("/kaggle/input/competitions/rsna-knee-abnormality-detection"),
              Path("/kaggle/input/rsna-knee-abnormality-detection"),
              Path("data"), Path(".")]:
        if (c / "test.csv").is_file() and (c / "test_series").is_dir():
            return c
    # last resort: two-level scan, because the mount is nested one deeper than usual
    base = Path("/kaggle/input")
    if base.is_dir():
        for depth1 in sorted(p for p in base.iterdir() if p.is_dir()):
            for cand in [depth1] + sorted(p for p in depth1.iterdir() if p.is_dir()):
                if (cand / "test.csv").is_file():
                    return cand
    raise FileNotFoundError(
        f"competition mount not found (cwd {Path.cwd()}); expected a directory holding "
        f"test.csv and test_series/")


def find_dinov2(variant="small"):
    """Locate a mounted DINOv2 checkpoint directory by variant name."""
    base = Path("/kaggle/input")
    if not base.is_dir():
        return None
    hits = []
    for root, dirs, files in os.walk(base):
        dirs[:] = [d for d in dirs if d not in ("train_series", "test_series")]
        if "config.json" in files and "dinov2" in root.lower():
            hits.append(Path(root))
    for h in hits:
        if variant in str(h).lower():
            return h
    return hits[0] if hits else None


LABEL_COLS = TARGETS + [t + "__conf" for t in TARGETS]


class LabelSourceError(RuntimeError):
    """Raised when the labels did not come from where this run intended.

    Every other failure in this file is better survived than reported: a run that dies
    after the cache is built has spent the expensive half and scores nothing, so the
    guard around `main` swallows it and leaves the benchmark file behind. This one is
    the exception. Training on the weaker labels does not look like a failure - it
    completes, writes a plausible submission, and differs only in a log line - so it has
    to stop the run rather than be absorbed by a guard designed for crashes.
    """


def find_label_table():
    """Locate a mounted table of pre-read report labels, if one is attached.

    The lexicon turns a report into labels by matching morphology, and its failure
    mode is silence: on a phrasing it does not carry it emits no opinion rather than a
    wrong one. Silence is measurable without any ground truth - for each (report,
    finding) pair, did anything match? - and that measurement says the misses are
    concentrated in particular languages rather than spread evenly, on findings a knee
    report almost always comments on.

    Enumerating morphology for nine languages is the wrong instrument for that. Reading
    the sentence is the right one, and a language model reads it. Against the annotated
    studies the difference is large and one-sided, so when such a table is mounted it is
    preferred; when it is not, the lexicon runs and the pipeline is unchanged. Both paths
    produce the same columns, so nothing downstream knows which one supplied them.
    """
    base = Path("/kaggle/input")
    cands = []
    if base.is_dir():
        for root, dirs, files in os.walk(base):
            dirs[:] = [d for d in dirs if d not in ("train_series", "test_series")]
            cands += [Path(root) / f for f in files if f.startswith("report_labels")
                      and f.endswith(".csv")]
    cands += [p for p in (Path("data/derived/report_labels_v2.csv"),) if p.is_file()]
    for c in cands:
        try:
            head = pd.read_csv(c, nrows=1)
        except Exception:
            continue
        if "StudyInstanceUID" in head.columns and all(t in head.columns for t in TARGETS):
            return c
    return None


def label_mount_attached():
    """True when an input directory was attached that is meant to carry a label table.

    The fallback below is deliberate and has to stay silent for a run with no table
    attached, because that is the ordinary case for anyone reading this notebook. It
    must not stay silent for the other case: a table was attached and could not be used.
    Those two are indistinguishable from the labels alone - both end with the lexicon -
    so they are separated here by whether the mount exists at all.
    """
    base = Path("/kaggle/input")
    if not base.is_dir():
        return False
    return any("label" in p.name.lower() for p in base.iterdir() if p.is_dir())


def read_labels(train_df):
    """Labels for every training study, from a mounted table or from the lexicon.

    Studies the mounted table does not cover fall back to the lexicon rather than being
    dropped, so a partial table degrades coverage instead of losing rows.
    """
    n = len(train_df)
    lab = pd.DataFrame([extract(r) for r in train_df["Report"].fillna("")])
    lab["StudyInstanceUID"] = train_df["StudyInstanceUID"].values
    lab = lab.set_index("StudyInstanceUID")

    src = find_label_table()
    if src is None:
        if label_mount_attached():
            raise LabelSourceError(
                "LABEL SOURCE: a label dataset is mounted but no usable table was found "
                "in it. Falling back to the lexicon here would train on the weaker "
                "labels and say so only in a log line, so the run stops instead.")
        log(f"LABEL SOURCE: lexicon, {n} studies (no table mounted)")
        return lab

    tab = pd.read_csv(src).set_index("StudyInstanceUID")
    missing = [c for c in LABEL_COLS if c not in tab.columns]
    if missing:
        raise LabelSourceError(
            f"LABEL SOURCE: {src} is missing {len(missing)} expected columns "
            f"(first: {missing[0]!r}). Refusing to fall back silently.")
    hit = lab.index.intersection(tab.index)
    if not len(hit):
        raise LabelSourceError(
            f"LABEL SOURCE: {src} shares no StudyInstanceUID with train.csv.")
    log(f"LABEL SOURCE: {src.name} covers {len(hit)} of {n} studies, "
        f"lexicon for the remaining {n - len(hit)}")
    lab.loc[hit, LABEL_COLS] = tab.loc[hit, LABEL_COLS].values
    return lab


ROOT = find_root()
log(f"input root: {ROOT}")


IMG = CACHE_IMG            # kept as the name the pixel reader and cache use


def available_gb():
    """Memory this machine will actually lend, read rather than assumed.

    A hardcoded ceiling is a guess about a machine the author is not sitting at, and a
    guess that is too low costs coverage silently while a guess that is too high ends the
    run. The machine will say, so it is asked.
    """
    try:
        with open("/proc/meminfo") as fh:
            info = {k.strip(): v for k, v in
                    (l.split(":", 1) for l in fh if ":" in l)}
        return int(info["MemAvailable"].split()[0]) / 1024 ** 2
    except Exception:
        return CACHE_BUDGET_GB / CACHE_FRACTION      # fall back to the old constant


def plan_cache(n_study, n_test=0):
    """Choose how many slices per slot the memory the machine has will allow.

    The cache is n_study x n_slot x slices x IMG^2 bytes. Coverage is the cheap axis -
    linear - and resolution the expensive one, so when the budget binds it is the slice
    count that gives way rather than the pixel grid. Deciding once, from the training
    corpus size, keeps train and test caches on the same group layout.

    Only a fraction of what is free is taken. The rest is not slack: the encoder, its
    activations, the pinned batches and the frames all come out of the same pool, and the
    cache is the one allocation big enough that overshooting it kills the run outright.
    """
    avail = available_gb()
    budget = min(avail * CACHE_FRACTION, CACHE_BUDGET_MAX_GB)
    # Both caches are held at once, and the test half is what the visible run cannot
    # show: here it is a handful of studies, and at scoring it is the whole hidden set.
    # Sizing against the training corpus alone therefore passes every run that can be
    # watched and overruns the one that counts.
    n_total = n_study + max(n_test, int(TEST_SHARE * n_study))
    per_slice = n_total * N_SLOT * IMG * IMG
    afford = int(budget * 1024 ** 3 // max(per_slice, 1))
    groups = max(1, min(N_GROUP_MAX, afford // GROUP))
    log(f"memory: {avail:.1f} GB available, {budget:.1f} GB to the cache; "
        f"sizing for {n_study} train + {n_total - n_study} test studies "
        f"-> {groups} group(s) of {GROUP} = {groups * GROUP} slices per slot"
        + (f" (wanted {N_GROUP_MAX})" if groups < N_GROUP_MAX else ""))
    return groups


N_GROUP = plan_cache(len(pd.read_csv(ROOT / "train.csv")),
                     len(pd.read_csv(ROOT / "test.csv")))
CACHE_SLICES = GROUP * N_GROUP
log(f"cache layout: {N_GROUP} groups x {GROUP} slices = {CACHE_SLICES} per slot")


In [10]:
HDR_TAGS = ["SeriesDescription", "SequenceName", "ScanOptions", "ScanningSequence",
            "RepetitionTime", "EchoTime", "Laterality", "PixelSpacing", "Rows",
            "Columns", "RescaleSlope", "RescaleIntercept",
            # Position and orientation are read from the same header probe() already
            # opens, so they cost nothing, and they are what recovers the side when the
            # Laterality tag is absent - which it is for half the studies here.
            "ImagePositionPatient", "ImageOrientationPatient"]


def _hdr_vec(s, n):
    """Parse a DICOM multi-value string as stored by probe(): floats joined by `|`."""
    if not isinstance(s, str):
        return None
    try:
        v = [float(x) for x in s.split("|")]
    except ValueError:
        return None
    return np.array(v) if len(v) >= n else None


def side_from_geometry(h):
    """Study -> 'L' / 'R' / None, from where the image sits in the patient.

    `Laterality` (0020,0060) is Type 2C and may legitimately be absent; in this corpus it
    is missing on exactly half the studies, and the vendors it is missing from are whole
    vendors rather than scattered series. A study with no tag is not a left knee, but the
    normalisation upstream treats it as one, so half the corpus was never normalised and
    the five side-defined targets - the two menisci, the two tibiofemoral compartments
    and the medial collateral ligament - saw that axis reversed on a large minority of it.

    The patient coordinate system fixes this without the tag: +x is the patient's left, so
    the centre of a right knee sits at negative x. The centre is used rather than
    `ImagePositionPatient` itself because that is the corner of the image, which is offset
    by half a field of view - enough to change the sign on a knee near the midline.

    The median over a study's series is what is thresholded, not a single series: probe()
    reads one arbitrary slice per series, which on a sagittal stack can sit anywhere
    across the joint. Studies whose centre falls near the midline are left unresolved
    rather than guessed - measured against the tagged half, the rule is right 97% of the
    time overall and no better than chance inside 20 mm.
    """
    cx = {}
    for r in h.itertuples(index=False):
        ipp = _hdr_vec(getattr(r, "ImagePositionPatient", None), 3)
        iop = _hdr_vec(getattr(r, "ImageOrientationPatient", None), 6)
        ps = _hdr_vec(getattr(r, "PixelSpacing", None), 2)
        rows, cols = getattr(r, "Rows", None), getattr(r, "Columns", None)
        if ipp is None or iop is None or ps is None or not rows or not cols:
            continue
        try:
            c = ipp[:3] + iop[:3] * ps[1] * float(cols) / 2 + iop[3:6] * ps[0] * float(rows) / 2
        except (TypeError, ValueError):
            continue
        cx.setdefault(r.StudyInstanceUID, []).append(float(c[0]))
    out = {}
    for st, xs in cx.items():
        m = float(np.median(xs))
        out[st] = None if abs(m) < LAT_MIN_OFFSET_MM else ("R" if m < 0 else "L")
    return out


def side_from_corner_x(h):
    """The laterality an imported member was fitted under.

    It thresholds the median raw `ImagePositionPatient` x over a study's series. That is
    the x of the image *corner*, not of its centre, so it differs from the rule above by
    up to half a field of view - which is enough to reverse the sign on a knee scanned
    near the midline. The dead zone is 5 mm rather than 20 mm, so it also commits on
    studies the rule above leaves unresolved.

    Neither difference changes a shape. Each one decides whether a study is mirrored, and
    a study mirrored one way at training and the other at inference presents the five
    side-defined targets with their axis reversed.
    """
    out = {}
    for st, g in h.groupby("StudyInstanceUID"):
        xs = []
        for r in g.itertuples(index=False):
            ipp = _hdr_vec(getattr(r, "ImagePositionPatient", None), 3)
            if ipp is not None and np.isfinite(ipp).all():
                xs.append(float(ipp[0]))
        if not xs:
            out[st] = None
            continue
        x = float(np.median(xs))
        # DICOM patient coordinates are LPS: +x is the patient's left.
        out[st] = None if abs(x) < LEGACY_LAT_OFFSET_MM else ("R" if x < 0 else "L")
    return out


def lat_of(h, tag=""):
    """Study -> 'L' / 'R' / None: the tag where it exists, geometry where it does not.

    The tag is present on exactly half the studies here and is sometimes an empty
    string rather than absent, which is not the same as NaN. Treating the other half
    as left-sided is what `normalise_laterality` did by omission, so the geometry
    fallback is not a refinement - it is the difference between normalising half the
    corpus and normalising all of it.
    """
    geo = side_from_corner_x(h) if RULES["lat"] == "corner_x" else side_from_geometry(h)
    d, n_tag, n_geo, n_none, n_disagree = {}, 0, 0, 0, 0
    for st, g in h.groupby("StudyInstanceUID"):
        v = [str(x).strip().upper() for x in g["Laterality"].dropna()]
        if RULES["lat"] == "corner_x" and "ImageLaterality" in g.columns:
            # The legacy rule reads the second tag too, so a study tagged only there is
            # resolved from the tag rather than from geometry.
            v += [str(x).strip().upper() for x in g["ImageLaterality"].dropna()]
        v = [x[0] for x in v if x and x[0] in ("L", "R")]
        side = v[0] if v else None
        if side is not None:
            n_tag += 1
            if geo.get(st) is not None and geo[st] != side:
                n_disagree += 1
        else:
            side = geo.get(st)
            n_geo += side is not None
            n_none += side is None
        d[st] = side
    log(f"{tag}laterality: {n_tag} from the tag, {n_geo} from geometry, "
        f"{n_none} unresolved; tag and geometry disagree on {n_disagree} "
        f"({n_disagree / max(n_tag, 1):.1%} of the tagged)")
    return d



def probe(item):
    split, study, series, path = item
    row = {"split": split, "StudyInstanceUID": study, "SeriesInstanceUID": series,
           "dir": path}
    try:
        files = sorted(e.name for e in os.scandir(path) if e.name.endswith(".dcm"))
        row["files"] = files
        row["n_slices"] = len(files)
        if not files:
            return row
        ds = pydicom.dcmread(os.path.join(path, files[len(files) // 2]),
                             stop_before_pixels=True, force=True)
        for t in HDR_TAGS:
            v = getattr(ds, t, None)
            if v is None:
                row[t] = None
            elif isinstance(v, (list, tuple)) or type(v).__name__ == "MultiValue":
                row[t] = "|".join(str(x) for x in v)
            else:
                row[t] = str(v)
    except Exception as exc:
        row["err"] = str(exc)[:120]
    return row


def walk(split):
    """Every series directory of a split, with one header read per series.

    An absent split returns an empty frame *with the columns annotate expects*. Returning
    a bare DataFrame looks like the same thing and is not: the next call indexes
    `SeriesDescription` and raises KeyError, so the branch that exists to survive a
    missing split is what turns it into a crash.
    """
    base = ROOT / split
    items = []
    if not base.is_dir():
        return pd.DataFrame(columns=["split", "StudyInstanceUID", "SeriesInstanceUID",
                                     "dir", "files", "n_slices"] + HDR_TAGS)
    for study in os.scandir(base):
        if study.is_dir():
            for series in os.scandir(study.path):
                if series.is_dir():
                    items.append((split, study.name, series.name, series.path))
    with ThreadPoolExecutor(max_workers=HDR_THREADS) as pool:
        rows = list(pool.map(probe, items))
    return pd.DataFrame(rows)


def annotate(df):
    """Recover fat suppression and pulse-sequence weighting from the header."""
    desc = (df["SeriesDescription"].fillna("") + " " + df["SequenceName"].fillna(""))
    desc = desc.str.lower().str.replace(_SEP, " ", regex=True)

    opts = df["ScanOptions"].fillna("").str.upper().str.split("|")
    # GE writes SAT_GEMS for spatial saturation, so ScanOptions must be matched as
    # exact tokens; a substring test on "SAT" fires on non-fat-sat series.
    opts_fs = opts.apply(lambda ts: any(t.strip() in FATSAT_OPTS for t in ts))
    df["fatsat"] = desc.str.contains(_FATSAT_RX) | opts_fs

    tr = pd.to_numeric(df["RepetitionTime"], errors="coerce")
    te = pd.to_numeric(df["EchoTime"], errors="coerce")
    gre = df["ScanningSequence"].fillna("").str.upper().str.contains("GR")
    t1, t2, pdw = desc.str.contains(_T1_RX), desc.str.contains(_T2_RX), desc.str.contains(_PD_RX)

    df["weight"] = np.where(t1 & ~t2 & ~pdw, "T1",
                     np.where(t2 & ~pdw, "T2",
                       np.where(pdw, "PD",
                         np.where(gre, "GRE",
                           np.where(tr < 800, "T1",
                             np.where(te > 60, "T2",
                               np.where(tr >= 800, "PD", "UNK")))))))
    df["fluid"] = np.isin(df["weight"], ["PD", "T2"])
    df["px"] = pd.to_numeric(
        df["PixelSpacing"].fillna("").str.split("|").str[0].replace("", np.nan),
        errors="coerce")
    return df


In [11]:
def pick_slots(series_df, plane_map):
    """One series per slot per study.

    Ties are broken toward the stack with the most slices: a thicker stack samples the
    joint more densely, and the three-slice sampler below benefits from the margin.
    """
    series_df = series_df.copy()
    series_df["plane"] = series_df["SeriesInstanceUID"].map(plane_map)
    out = {}
    for study, g in series_df.groupby("StudyInstanceUID"):
        chosen = {}
        for name, plane, fluid, fs in SLOTS:
            sel = (g["plane"] == plane) & (g["fatsat"] == fs)
            # fluid=None means "do not condition on weighting" - the public scheme,
            # where the single provided flag stands in for both axes at once.
            if fluid is not None:
                sel &= (g["fluid"] == fluid)
            cand = g[sel]
            # A slot with no series matching its predicate stays empty, and no substitute
            # is admitted from a neighbouring predicate. Relaxing the weighting to fill a
            # T1 slot would draw from the pool `SAG_FLUID_NOFS` selects from, since that
            # pool is what remains once the weighting is dropped: over the training corpus
            # it would put one series in two slots for 2383 of 4407 studies and leave 56%
            # of the T1 slot holding PD or T2. The presence mask would then assert a
            # sequence that was never acquired, and the per-diagnosis softmax of ยง6 would
            # divide its attention across two identical slots, giving one acquisition
            # about twice the weight it carries in a study that holds both. The mask is
            # there to say a slot is absent, which is what an absent slot is.
            if len(cand) == 0 and RULES["slot_fallback"] and fluid is False:
                # The relaxation the paragraph above rejects, reproduced because an
                # imported member was fitted with its T1 slots filled this way: over half
                # of that member's training studies had a T1 slot holding a series that
                # is not T1. Leaving those slots empty would present it with a presence
                # mask it never saw.
                cand = g[(g["plane"] == plane) & (~g["fatsat"])]
            if len(cand):
                chosen[name] = cand.sort_values("n_slices", ascending=False).iloc[0]
        out[study] = chosen
    return out


In [12]:
ORDER_TAGS = [(0x0020, 0x0032), (0x0020, 0x0037), (0x0020, 0x0013)]

# Series in which at least one sampled slice would not decode. A list rather than a
# counter because appending is atomic under the reader threads, and reported rather than
# swallowed: unreported, a decode failure is indistinguishable from a black knee.
DECODE_FAILED = []


def cache_tag(rules=None):
    """The name a decoded cache is stored under.

    It has to name everything that decides the pixels, not only their dimensions. Two
    configurations that agree on resolution, slice count, crop and band but disagree on
    how a slice is chosen produce different arrays of identical shape - so a tag built
    from the dimensions alone lets the second attach to the first one's file and train
    against pixels it never asked for, with nothing anywhere reporting a mismatch.

    A native reading keeps the plain name, so caches decoded before the rules existed
    stay valid; anything else earns a suffix.
    """
    r = dict(RULES if rules is None else rules)
    t = (f"{CACHE_IMG}px_{CACHE_SLICES}sl_{int(CROP_MM)}mm_"
         f"{SLICE_BAND[0]:.2f}-{SLICE_BAND[1]:.2f}")
    if {k: r.get(k, v) for k, v in RULES_NATIVE.items()} != RULES_NATIVE:
        t += "_" + hashlib.md5(json.dumps(r, sort_keys=True).encode()).hexdigest()[:6]
    return t


def _natural_key(name):
    return tuple(int(x) if x.isdigit() else x.lower()
                 for x in re.split(r"(\d+)", str(name)))


def _order_dominant_axis(rec):
    """The slice order an imported member was fitted under.

    It sorts on the raw patient coordinate along whichever axis varies most across the
    stack, rather than on the projection onto the slice normal. The two differ by a sign,
    not by a formula: measured over this corpus every sagittal series has a slice normal
    with n_x in [-1.00, -0.98], so p.n is the negative of the raw x this sorts on and the
    two stacks come out exactly reversed. Because the band sampler truncates rather than
    rounds, its nine indices are not symmetric about the middle, so nine slices drawn from
    a twenty-six slice stack under one order share two with the other.

    Missing geometry falls back to `InstanceNumber` and then to a natural sort of the file
    name, both at the same 80% threshold the imported pipeline used.
    """
    files, d = rec["files"], rec["dir"]
    rows = []
    for pos, f in enumerate(files):
        ipp = inst = None
        try:
            ds = pydicom.dcmread(os.path.join(d, f), force=True, stop_before_pixels=True,
                                 specific_tags=["ImagePositionPatient", "InstanceNumber"])
            raw = getattr(ds, "ImagePositionPatient", None)
            if raw is not None and len(raw) >= 3:
                c = np.asarray(raw[:3], dtype=np.float64)
                if np.isfinite(c).all():
                    ipp = c
            n = getattr(ds, "InstanceNumber", None)
            if n is not None:
                inst = float(n)
        except Exception:
            pass
        rows.append((f, ipp, inst, pos))

    placed = [r for r in rows if r[1] is not None]
    need = max(2, int(0.8 * len(rows)))
    if len(placed) >= need:
        xyz = np.stack([r[1] for r in placed])
        axis = int(np.argmax(np.ptp(xyz, axis=0)))
        spare = float(np.nanmedian(xyz[:, axis]))
        rows.sort(key=lambda r: (float(r[1][axis]) if r[1] is not None else spare,
                                 r[2] if r[2] is not None else float("inf"), r[3]))
    elif sum(r[2] is not None for r in rows) >= need:
        rows.sort(key=lambda r: (r[2] if r[2] is not None else float("inf"), r[3]))
    else:
        rows.sort(key=lambda r: _natural_key(r[0]))
    return [r[0] for r in rows], True


def order_slices(rec):
    """Return the series' files sorted along the through-plane axis.

    A DICOM file name here is a SOP Instance UID, which is assigned arbitrarily. Sorting
    by it therefore produces an order uncorrelated with anatomy - measured over one
    series, Spearman between file-name rank and physical position is 0.009, i.e. none.
    Anything that assumes the file order means something is then operating on noise: the
    three channels of a "2.5D" input are three unrelated views rather than neighbouring
    slices, "the middle of the stack" is a random subset, and reversing slice order to
    normalise laterality reverses nothing meaningful.

    The physical order is recoverable exactly. Each slice carries its position in patient
    coordinates and the in-plane axes; projecting the position onto the slice normal
    gives a signed through-plane coordinate, monotonic along the stack:

        n = r_x  x  r_y ,      k = p . n

    `InstanceNumber` is the fallback. It usually tracks the projection up to sign, but
    interleaved and multi-echo acquisitions need not number slices in the order they
    occupy in space - but the projection is signed in patient
    coordinates, which is what laterality normalisation needs.
    """
    if RULES["order"] == "dominant_axis":
        return _order_dominant_axis(rec)
    files, d = rec["files"], rec["dir"]
    keyed = []
    for f in files:
        k = None
        try:
            ds = pydicom.dcmread(os.path.join(d, f), force=True, stop_before_pixels=True,
                                 specific_tags=ORDER_TAGS)
            iop = np.asarray(ds.ImageOrientationPatient, dtype=float)
            ipp = np.asarray(ds.ImagePositionPatient, dtype=float)
            k = float(np.dot(ipp, np.cross(iop[:3], iop[3:])))
        except Exception:
            try:
                k = float(ds.InstanceNumber)
            except Exception:
                k = None
        keyed.append((k, f))
    if any(k is None for k, _ in keyed):
        # A series with no usable geometry keeps its arbitrary order; that is worse than
        # sorting but better than dropping the series, and it is logged as a count.
        return files, False
    return [f for _, f in sorted(keyed, key=lambda t: t[0])], True


def read_slot(rec, n_slice=None, out_size=None):
    """`n_slice` physically spread slices from one series, at `out_size` pixels.

    Returns uint8 [n_slice, out, out] normalised per-series to its 1st-99th
    percentile. Percentiles rather than min/max because MR intensity has no absolute
    scale and a single bright vessel would otherwise compress the whole dynamic range.

    Reading is the expensive half of this pipeline, so the caller reads once at the
    largest configuration it needs and derives the smaller ones from the returned buffer
    rather than re-reading.
    """
    n_slice = GROUP if n_slice is None else n_slice
    out_size = IMG if out_size is None else out_size
    files, d, px = rec.get("ordered") or rec["files"], rec["dir"], rec["px"]
    n = len(files)
    if n == 0:
        return None
    # Spread the samples over a central band of the stack: the outermost slices of a knee
    # series are mostly soft tissue outside the joint. The band is a constant rather than
    # a literal because how much of the stack is worth reading depends on how many slices
    # are being taken - at three the middle is all that fits, while at sixteen the ends
    # are worth having, and a Baker cyst sits at the posteromedial end of a sagittal one.
    lo, hi = int(SLICE_BAND[0] * (n - 1)), int(SLICE_BAND[1] * (n - 1))
    idx = np.unique(np.linspace(lo, hi, n_slice).astype(int)) if hi > lo else np.array([n // 2])
    while len(idx) < n_slice:
        idx = np.append(idx, idx[-1])

    planes = []
    for i in idx[:n_slice]:
        try:
            ds = pydicom.dcmread(os.path.join(d, files[int(i)]), force=True)
            a = ds.pixel_array.astype(np.float32)
            sl = float(getattr(ds, "RescaleSlope", 1) or 1)
            ic = float(getattr(ds, "RescaleIntercept", 0) or 0)
            a = a * sl + ic
        except Exception:
            a = None                      # no shape is known here; see below
        planes.append(a)

    # A slice that would not decode has no shape of its own, and inventing one is how a
    # single unreadable file erases a whole series: a substitute allocated at the resize
    # target while the decoded slices are still native makes the shape check below take
    # the substitute as the authority and zero the good slices with it, leaving a black
    # slot that the presence mask still reports as acquired.
    #
    # A failure is instead filled from the nearest slice that did decode - the same
    # convention the sampler already uses when the band holds fewer distinct slices than
    # were asked for - and a series where nothing decodes is reported absent, which the
    # mask can express, rather than black, which it cannot.
    got = [k for k, p in enumerate(planes) if p is not None]
    if RULES["decode_fill"] == "zero":
        # What an imported member was fitted with: a failure becomes a zero plane at the
        # resize target, which the shape check below then propagates to the whole slot.
        # It is the behaviour the paragraph above describes and rejects, kept here only
        # because that member's weights were learned against slots blacked out this way.
        if not got:
            DECODE_FAILED.append(rec.get("SeriesInstanceUID", d))
        planes = [np.zeros((out_size, out_size), np.float32) if p is None else p
                  for p in planes]
        got = list(range(len(planes)))
    if not got:
        DECODE_FAILED.append(rec.get("SeriesInstanceUID", d))
        return None
    if len(got) < len(planes):
        DECODE_FAILED.append(rec.get("SeriesInstanceUID", d))
        for k, p in enumerate(planes):
            if p is None:
                planes[k] = planes[min(got, key=lambda j: abs(j - k))]

    # Slices of one series can still differ in matrix size - multi-echo and some
    # reformats do - and those are genuinely not stackable.
    shp = planes[0].shape
    planes = [p if p.shape == shp else np.zeros(shp, np.float32) for p in planes]
    vol = np.stack(planes)

    # constant physical extent, then resize: PixelSpacing varies 3.4x across the corpus
    if px and np.isfinite(px) and px > 0:
        want = int(round(CROP_MM / px))
        h, w = shp
        if 16 < want < min(h, w):
            cy, cx = h // 2, w // 2
            half = want // 2
            vol = vol[:, max(0, cy - half):cy + half, max(0, cx - half):cx + half]

    lo_v, hi_v = np.percentile(vol, [1, 99])
    vol = np.clip((vol - lo_v) / max(hi_v - lo_v, 1e-6), 0, 1)

    t = torch.from_numpy(np.ascontiguousarray(vol)).unsqueeze(0)
    t = F.interpolate(t, size=(out_size, out_size), mode="bilinear", align_corners=False)
    # uint8, not float32. These buffers queue up between the reader threads and the
    # encoder, and at this size a float32 slot-series is several megabytes. Intensity is
    # already normalised into [0, 1] here, so eight bits cost nothing that a bilinear
    # resize has not already cost, and the queue is a quarter the size.
    return (t.squeeze(0) * 255).round().clamp(0, 255).to(torch.uint8)


In [13]:
def normalise_laterality(img, plane, lat):
    """Map every knee onto a left-knee convention.

    Coronal and axial views mirror under a horizontal flip. Sagittal stacks are not
    mirror images of each other - the slice order runs medial-to-lateral in opposite
    directions - so the channel order is reversed instead.
    """
    if lat != "R":
        return img
    if plane in ("Coronal", "Axial"):
        return torch.flip(img, dims=[-1])
    return torch.flip(img, dims=[0])


In [14]:
# Where the geometric slice order may be remembered between runs. Unset on the platform,
# because each run gets a fresh machine and there is nothing to remember; set off it,
# where the same corpus is cached again at every resolution and slice count and the order
# is a function of neither. It is opt-in so that the scored run's behaviour is decided by
# the code rather than by whether a file happens to be lying about.
ORDER_CACHE = os.environ.get("RSNA_ORDER_CACHE") or None


def build_cache(slot_map, plane_map, lat_map, tag):
    """Decode every (study, slot) once into an in-memory uint8 array.

    Fine-tuning revisits the same pixels every epoch. Reading them from the mount each
    time would make the epoch count a function of I/O rather than of learning, so they
    are decoded once and held as bytes: intensity has already been normalised into
    [0, 1], and eight bits cost nothing a bilinear resize has not already cost.

    CACHE_SLICES positions are kept per slot, which the training loop reads as N_GROUP
    groups of GROUP consecutive channels.
    """
    studies = sorted(slot_map)
    sidx = {s: i for i, s in enumerate(studies)}
    cache = np.zeros((len(studies), N_SLOT, CACHE_SLICES, IMG, IMG), np.uint8)
    mask = np.zeros((len(studies), N_SLOT), np.float32)
    log(f"{tag}: cache {cache.shape} = {cache.nbytes / 1024 ** 3:.1f} GB")

    jobs = [(st, k, plane, slot_map[st][name])
            for st in studies
            for k, (name, plane, _, _) in enumerate(SLOTS)
            if name in slot_map[st]]
    n_job = len(jobs)

    # Ordering first, and as its own pass. It reads one header per slice of every chosen
    # series - far more file opens than the decode that follows - and on a network mount
    # that is latency, not work, so it gets its own wider pool.
    t_ord = time.time()
    n_slice_total = sum(len(j[3]["files"]) for j in jobs)
    log(f"{tag}: ordering {len(jobs)} slot-series ({n_slice_total} slice headers)")
    ok = done = 0
    CHUNK_O = 1024

    # A remembered order, when one is offered. The projection depends on the DICOM
    # geometry alone, so it is the same at every resolution and every slice count, and
    # it costs one header read per slice - the largest single cost in this pass. An entry
    # is validated by the number of files present, so a tree that has changed under it is
    # recomputed rather than trusted: order is derived data, and a stale entry would be
    # invisible in the way that matters most.
    seen = {}
    if ORDER_CACHE and Path(ORDER_CACHE).is_file():
        try:
            import json as _json
            seen = _json.loads(Path(ORDER_CACHE).read_text())
        except (OSError, ValueError):
            seen = {}
        hit = 0
        for _, _, _, rec in jobs:
            e = seen.get(rec["SeriesInstanceUID"])
            if e and len(e["files"]) == len(rec["files"]):
                rec["ordered"] = e["files"]
                ok += int(e["good"])
                hit += 1
        jobs = [j for j in jobs if "ordered" not in j[3]]
        log(f"{tag}: {hit} slot-series ordered from {ORDER_CACHE}, {len(jobs)} to read")

    with ThreadPoolExecutor(max_workers=ORDER_THREADS) as pool:
        for c0 in range(0, len(jobs), CHUNK_O):
            block = jobs[c0:c0 + CHUNK_O]
            for (_, _, _, rec), (files, good) in zip(
                    block, pool.map(lambda j: order_slices(j[3]), block)):
                rec["ordered"] = files
                ok += int(good)
                done += 1
                if ORDER_CACHE:
                    seen[rec["SeriesInstanceUID"]] = {"files": files, "good": bool(good)}
            # The ceiling is whichever comes first: the pass's own budget, or the share
            # of what is left of the run that it may take. The second is what makes the
            # first safe to set generously - a mount slow enough to matter cannot spend
            # the training time, because the budget shrinks as the run does.
            budget = min(ORDER_BUDGET_S, max(60.0, (TIME_BUDGET - (time.time() - T0)) * 0.35))
            if time.time() - t_ord > budget:
                log(f"{tag}: ordering budget spent at {done}/{len(jobs)}; "
                    f"the rest keep file order")
                break
    if ORDER_CACHE and done:
        import json as _json
        _t = Path(ORDER_CACHE).with_suffix(".tmp")
        _t.write_text(_json.dumps(seen))
        _t.replace(Path(ORDER_CACHE))
    log(f"{tag}: ordered {ok}/{n_job} by geometry "
        f"({n_job - ok} kept arbitrary) in {time.time() - t_ord:.0f}s")

    jobs = [(st, k, plane, slot_map[st][name])
            for st in studies
            for k, (name, plane, _, _) in enumerate(SLOTS)
            if name in slot_map[st]]
    log(f"{tag}: decoding {len(jobs)} slot-series")
    n_failed_before = len(DECODE_FAILED)

    CHUNK = 512
    done = 0
    with ThreadPoolExecutor(max_workers=PIX_THREADS) as pool:
        for c0 in range(0, len(jobs), CHUNK):
            block = jobs[c0:c0 + CHUNK]
            for (st, k, plane, _), img in zip(
                    block, pool.map(lambda j: read_slot(j[3], CACHE_SLICES, IMG), block)):
                done += 1
                if img is None:
                    continue
                cache[sidx[st], k] = normalise_laterality(img, plane,
                                                          lat_map.get(st)).numpy()
                mask[sidx[st], k] = 1.0
            if done % 4096 < CHUNK:
                log(f"  {tag} {done}/{len(jobs)}")
            if time.time() - T0 > TIME_BUDGET:
                log(f"  {tag}: time budget reached during decode")
                break
    n_failed = len(DECODE_FAILED) - n_failed_before
    log(f"{tag}: {int(mask.sum())}/{len(jobs)} slots filled"
        + (f"; {n_failed} series had a slice that would not decode" if n_failed else ""))
    gc.collect()
    return studies, cache, mask


In [15]:
class SlotHead(nn.Module):
    """Per-diagnosis attention over the slot embeddings of one study.

    Each finding is read on particular sequences - cruciates sagittally, collateral
    ligaments and the meniscal body coronally, patellar cartilage axially - so pooling
    the slots identically would dilute the one that carries the evidence with the rest.

    The aggregation is deliberately this simple. With a study-level label there is no
    signal telling the model which part of a study matters, so extra attention
    parameters below the slot level would have nothing to learn from and would spend
    their capacity fitting noise.
    """

    def __init__(self, dim, n_slot, n_out, hidden=256, p=0.2, prior=False):
        super().__init__()
        self.proj = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, hidden), nn.GELU())
        self.slot_emb = nn.Parameter(torch.randn(n_slot, hidden) * 0.02)
        self.query = nn.Parameter(torch.randn(n_out, hidden) * 0.02)
        self.drop = nn.Dropout(p)
        self.out = nn.Linear(hidden, n_out)
        self.hidden = hidden
        # An imported member carries a fixed per-(diagnosis, slot) tilt on the attention
        # logits, set from the anatomy table below rather than learned. It is a buffer, so
        # it travels in the state dict and must exist for that member to load; exp(0.55)
        # gives a preferred slot about 1.73x the weight of an unpreferred one, which
        # biases the softmax without ever excluding a slot.
        p_ = torch.zeros(n_out, n_slot)
        if prior and n_slot == len(SLOTS) and n_out == len(TARGETS):
            for t, slots in SLOT_PRIOR_TABLE.items():
                if t in TARGETS:
                    p_[TARGETS.index(t), list(slots)] = SLOT_PRIOR_STRENGTH
        self.prior = prior
        if prior:
            self.register_buffer("slot_prior", p_)

    def forward(self, x, mask):
        h = self.proj(x) + self.slot_emb
        att = torch.einsum("bsh,oh->bos", h, self.query) / self.hidden ** 0.5
        if self.prior:
            att = att + self.slot_prior.unsqueeze(0)
        att = att.masked_fill(mask.unsqueeze(1) < 0.5, -1e4).softmax(-1)
        ctx = self.drop(torch.einsum("bos,bsh->boh", att, h))
        return (ctx * self.out.weight.unsqueeze(0)).sum(-1) + self.out.bias


In [16]:
class Model(nn.Module):
    """Encoder plus head, trained end to end.

    A study arrives as a bag of slot images. The bag is flattened for the encoder and
    folded back before the head, so the encoder never sees the study structure and the
    head never sees pixels.
    """

    def __init__(self, backbone, dim, pool="cls_mean", prior=False):
        super().__init__()
        self.backbone = backbone
        self.pool = pool
        self.head = SlotHead(dim * POOL_PARTS[pool], N_SLOT, len(TARGETS), prior=prior)
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def forward(self, imgs, mask, img_size=None):
        B, S = imgs.shape[:2]
        x = imgs.reshape(B * S, *imgs.shape[2:]).float().div_(255.0)
        if img_size is not None and img_size != x.shape[-1]:
            # The cache is held at the highest resolution any configuration needs; the
            # rest downsample from it, so every configuration sees the same pixels
            # through a different sampling grid rather than a different crop.
            x = F.interpolate(x, size=(img_size, img_size), mode="bilinear",
                              align_corners=False)
        x = (x - self.mean) / self.std
        out = self.backbone(pixel_values=x).last_hidden_state
        patch = out[:, 1:]
        parts = [out[:, 0], patch.mean(1)]
        if self.pool == "cls_mean_focal":
            # The upper tail of each channel over the patch grid, taken per channel
            # rather than by selecting whole patches: a finding occupies a small part of
            # the field, so a plain mean over 256 patches dilutes it by two orders of
            # magnitude, and this keeps the top eighth of each channel's responses.
            k = max(1, patch.shape[1] // 8)
            parts.append(patch.topk(k, dim=1).values.mean(1))
        feat = torch.cat(parts, dim=1).reshape(B, S, -1)
        return self.head(feat, mask)


In [17]:
def build_model(unfreeze_last, source=None, variant="small", pool="cls_mean",
                prior=False):
    """Load the encoder and open the last `unfreeze_last` blocks for training.

    The early blocks of a self-supervised transformer are generic edge and texture
    filters; the late blocks carry semantics. Opening only the late ones is the cautious
    choice - there may not be enough supervision here to improve the early ones and there
    is certainly enough to damage them - but how far the line should sit is a question
    the corpus has to answer rather than the intuition.

    `source` names where the weights come from. Left unset it is the attached model
    directory, which is the only thing available here. It is a parameter so that a run
    off the platform builds the same object from the same code rather than from a second
    definition that has to be kept in step by hand.
    """
    from transformers import AutoModel
    p = source if source is not None else find_dinov2(variant)
    if p is None:
        raise FileNotFoundError("DINOv2 weights not attached")
    bb = AutoModel.from_pretrained(str(p))
    n_layer = len(bb.encoder.layer)
    for prm in bb.parameters():
        prm.requires_grad = False
    for blk in bb.encoder.layer[max(0, n_layer - unfreeze_last):]:
        for prm in blk.parameters():
            prm.requires_grad = True
    for prm in bb.layernorm.parameters():
        prm.requires_grad = True
    dim = bb.config.hidden_size
    trainable = sum(p.numel() for p in bb.parameters() if p.requires_grad)
    log(f"backbone: {n_layer} blocks, last {unfreeze_last} trainable "
        f"({trainable / 1e6:.1f}M params), feature dim {dim * POOL_PARTS[pool]}")
    return Model(bb, dim, pool=pool, prior=prior)


In [18]:
FINGERPRINT_TOL = 2e-3


def fingerprint(model, dev, img_size, n_slot=None, group=None, seed=None):
    """The model's output on a fixed synthetic bag, as a portable identity.

    Weights that are loaded but read through the wrong preprocessing produce predictions,
    not errors. The submission is well formed, the log says nothing, and the difference is
    a number no output of the run reveals. Scaling that never happens, or happens twice,
    is enough on its own and changes no shape anywhere.

    So a set of weights carries the answer it gave to a question with no data in it. The
    input is generated from a seed rather than read, so it is the same on any machine, and
    it is pushed through the whole forward path - the byte scaling, the ImageNet
    normalisation, the resize, the encoder, the slot attention. Any of those differing
    moves the output by order one. Numerics differing between two GPUs moves it by about
    1e-5, which is why the tolerance sits between them rather than at zero.

    This checks that the model computes what it computed when it was fitted. It cannot
    check that the pixels reaching it are the right pixels; `read_slot` and the header
    pass answer to their own tests.
    """
    n_slot = N_SLOT if n_slot is None else n_slot
    group = GROUP if group is None else group
    seed = SEED if seed is None else seed
    g = torch.Generator().manual_seed(seed)
    imgs = torch.randint(0, 256, (2, n_slot, group, img_size, img_size),
                         generator=g, dtype=torch.uint8).to(dev)
    mask = torch.ones(2, n_slot, device=dev)
    mask[1, -1] = 0.0                       # exercise the masked branch of the softmax
    was_training = model.training
    model.eval()
    with torch.no_grad():
        # float32 throughout: autocast would make the value depend on which device
        # happened to run it, and the point of the number is that it does not.
        out = model(imgs, mask, img_size).float().cpu().numpy()
    if was_training:
        model.train()
    return out


def check_fingerprint(model, dev, img_size, expected, tol=FINGERPRINT_TOL, tag=""):
    """Compare against a stored fingerprint; raise when the model is not the same map."""
    got = fingerprint(model, dev, img_size)
    exp = np.asarray(expected, np.float32)
    if got.shape != exp.shape:
        raise WeightsError(f"{tag}fingerprint shape {got.shape} != stored {exp.shape}: "
                           f"the architecture is not the one these weights were fitted to")
    d = float(np.abs(got - exp).max())
    if d > tol:
        raise WeightsError(
            f"{tag}fingerprint differs by {d:.4g} (tolerance {tol:g}). The weights load "
            f"but do not compute what they computed when fitted - preprocessing, "
            f"resolution or architecture has moved between the two runs.")
    log(f"{tag}fingerprint matches within {d:.2g}")
    return d


class WeightsError(RuntimeError):
    """Raised when attached weights cannot be trusted to be the ones that were fitted.

    Deliberately fatal for the same reason as LabelSourceError: a run that predicts from
    a mismatched model completes, writes a plausible submission, and differs from a
    correct one only in a number no output of the run reveals.
    """


def find_weights(name="manifest.json"):
    """Locate a mounted weights package, or return None if none is attached.

    Same shape as `find_label_table`: the notebook must keep working for a reader who
    attaches nothing, so absence is a path rather than an error. What must not be silent
    is a package that is attached and unusable, and that is what `load_weights` refuses.
    """
    import json
    base = Path("/kaggle/input")
    if not base.is_dir():
        return None
    for root, dirs, files in os.walk(base):
        dirs[:] = [d for d in dirs if d not in ("train_series", "test_series")]
        if name not in files:
            continue
        # The manifest decides, not the filenames beside it. Testing for a naming
        # convention makes the search agree with whatever the packager happened to call
        # its files last, which is a second definition of what a package is.
        try:
            man = json.loads((Path(root) / name).read_text())
        except (OSError, ValueError):
            continue
        if isinstance(man.get("members"), list) and man["members"]:
            missing = [m["file"] for m in man["members"]
                       if not (Path(root) / m["file"]).is_file()]
            if missing:
                raise WeightsError(
                    f"{root} holds a manifest listing {len(man['members'])} members but "
                    f"{len(missing)} of their files are absent (first {missing[0]!r})")
            return Path(root)
    return None


# How a member is read at inference. Overlapping windows over the slices the cache
# already holds cost forward passes and no extra decoding, which is the cheap direction
# to spend; and averaging probabilities rather than logits is an arithmetic mean of risk
# rather than a geometric mean of odds, which orders studies differently. Both were
# chosen by measuring them on the folds each member held out rather than by argument.
TTA_OVERLAP = True
TTA_POOL = "prob"

# Public-frontier target pooling: the 0.899 notebook keeps the strongest window for
# focal findings, and averages the two strongest windows for ACL/MCL. Its public
# Pilkwang family is the same 20-member package used here, so these pooling decisions
# transfer directly. Preserve the independently validated original-view Synovitis
# route while retaining jitter for every other target.
PUBLIC_FRONTIER_TARGET_POOL = {
    "Fracture": "max",
    "Contusion": "max",
    "Medial Meniscus": "max",
    "Lateral Meniscus": "max",
    "ACL": "top2",
    "MCL": "top2",
    "Baker's": "max",
}
TTA_TARGET_POOL = {**PUBLIC_FRONTIER_TARGET_POOL, "Synovitis": "original_mean"}

# V25 uses the 58 complete annotation rows from the official train.csv, joined by
# StudyInstanceUID to the OOF predictions. The older y_derived artifact disagrees with
# those expert labels in 145 cells and is not used for this decision. Repeated
# target-stratified four-fold selection is macro-positive in 98% of 200 partitions and
# isolates four targets; every other target returns to the native family. Lateral
# Meniscus is shrunk from the selected median 1.00 to 0.75.
# Convert desired final fraction f to each of four legacy-fold member weights:
#     4*w/(20+4*w)=f -> w=5*f/(1-f).
LEGACY_MEMBER_WEIGHT_BY_TARGET = {
    "Lateral Meniscus": 15.0,           # final legacy fraction 0.75
    "Medial OA": 2.5,                   # final legacy fraction 1/3
    "Lateral OA": 15.0,                 # final legacy fraction 0.75
    "Contusion": 5.0,                   # final legacy fraction 0.50
}


def window_starts(n_slice, group, overlap=None):
    """Where each TTA window begins."""
    overlap = TTA_OVERLAP if overlap is None else overlap
    if overlap and n_slice >= group:
        return list(range(n_slice - group + 1))
    return [g * group for g in range(max(n_slice // group, 1))]


def apply_target_window_pool(values, probs, logits, original_probs, mapping, target_idx):
    """Apply a target-specific pooling map to one batch in place."""
    for target, mode in mapping.items():
        j = target_idx[target]
        if mode == "max":
            values[:, j] = probs[:, :, j].max(0).values
        elif mode == "mean":
            values[:, j] = probs[:, :, j].mean(0)
        elif mode == "logit_mean":
            values[:, j] = torch.sigmoid(logits[:, :, j].mean(0))
        elif mode == "original_mean":
            values[:, j] = original_probs[:, :, j].mean(0)
        elif mode in ("top2", "top3"):
            k = min(int(mode[3:]), probs.shape[0])
            values[:, j] = probs[:, :, j].topk(k, dim=0).values.mean(0)
        else:
            raise ValueError(f"unknown TTA pooling mode for {target}: {mode}")
    return values


@torch.no_grad()
def predict_member(model, cache, mask, idx, dev, img_size, group=None, pool=None,
                   starts=None, jitter=False, jitter_seed=SEED,
                   return_public_frontier=False):
    """Predict one member; pool views within windows and windows across a study."""
    group = GROUP if group is None else group
    pool = TTA_POOL if pool is None else pool
    starts = window_starts(cache.shape[2], group) if starts is None else list(starts)
    if not starts:
        raise ValueError("predict_member was given no windows to average over")
    target_idx = {t: j for j, t in enumerate(TARGETS)}
    unknown = (set(TTA_TARGET_POOL) | set(PUBLIC_FRONTIER_TARGET_POOL)) - set(target_idx)
    if unknown:
        raise ValueError(f"unknown target(s) in TTA_TARGET_POOL: {unknown}")

    jitter_gen = torch.Generator(device=dev)
    jitter_gen.manual_seed(int(jitter_seed) % (2**63 - 1))
    model.eval()
    out, public_frontier_out = [], []
    for b in range(0, len(idx), EVAL_BATCH):
        sel = idx[b:b + EVAL_BATCH]
        m = torch.from_numpy(mask[sel]).to(dev)
        win_probs, win_logits, win_original_probs = [], [], []
        for st in starts:
            rows = torch.from_numpy(
                np.ascontiguousarray(cache[sel, :, st:st + group])).to(dev)
            views = [rows] + ([augment(rows, generator=jitter_gen)] if jitter else [])
            view_probs, view_logits = [], []
            for view in views:
                with torch.autocast("cuda", enabled=dev.type == "cuda"):
                    z = model(view, m, img_size).float()
                view_logits.append(z)
                view_probs.append(torch.sigmoid(z))
            win_logits.append(torch.stack(view_logits).mean(0))
            win_probs.append(torch.stack(view_probs).mean(0))
            # views[0] is always the unaugmented acquisition. Preserve it so a
            # target can opt out of jitter without another encoder forward pass.
            win_original_probs.append(view_probs[0])

        probs = torch.stack(win_probs)      # [window, batch, target]
        logits = torch.stack(win_logits)
        original_probs = torch.stack(win_original_probs)
        v = (torch.sigmoid(logits.mean(0)) if pool == "logit" else probs.mean(0))
        v = apply_target_window_pool(
            v, probs, logits, original_probs, TTA_TARGET_POOL, target_idx
        )
        out.append(v.cpu().numpy())
        if return_public_frontier:
            public_v = apply_target_window_pool(
                original_probs.mean(0), original_probs, logits, original_probs,
                PUBLIC_FRONTIER_TARGET_POOL, target_idx,
            )
            public_frontier_out.append(public_v.cpu().numpy())
    primary = (np.concatenate(out) if out else
               np.zeros((0, len(TARGETS)), np.float32))
    if not return_public_frontier:
        return primary
    public_frontier = (np.concatenate(public_frontier_out) if public_frontier_out else
                       np.zeros((0, len(TARGETS)), np.float32))
    return primary, public_frontier


# HF from_pretrained mutates process-global state and is not guaranteed thread-safe, so
# model construction, weight loading and the fingerprint check are serialised; only
# inference -- the expensive part -- runs on both devices at once.
BUILD_LOCK = threading.Lock()
STATE_LOCK = threading.Lock()

# An independently trained four-fold bundle joins the vote at reduced weight: a second
# training run (public 0.836 on its own) adds decorrelated errors, which is the only thing an
# inference-only run can add that the 20-member package does not already have. 0.5 per
# fold puts the bundle at ~11% of the total vote -- roughly its quality gap.
LEGACY_BUNDLE_FILE = "rsna_20260807_v1.pt"
LEGACY_WEIGHT = 0.5


def find_legacy_bundle():
    base = Path("/kaggle/input")
    if not base.is_dir():
        return None
    for root, dirs, files in os.walk(base):
        dirs[:] = [d for d in dirs if d not in ("train_series", "test_series")]
        if LEGACY_BUNDLE_FILE in files:
            return Path(root) / LEGACY_BUNDLE_FILE
    return None


def legacy_group_members():
    """The four-fold bundle as extra, lower-weight members under RULES_LEGACY.

    The package's legacy pixel rules exist precisely to reproduce what this bundle was
    fitted on (dominant-axis slice order, corner-x laterality at 5 mm, T1 slot fallback,
    zero decode fill, 160 mm crop, central 60% band). The bundle predates fingerprints,
    which is accepted loudly and priced into its reduced weight; a fold whose state dict
    does not load, or whose predictions are degenerate, is dropped and costs its own
    vote only.
    """
    p = find_legacy_bundle()
    if p is None:
        log("no legacy bundle attached; blending skipped")
        return {}
    try:
        b = torch.load(p, map_location="cpu", weights_only=False)
        folds = b.get("fold_states") or []
        b_slots = [tuple(s)[0] for s in b.get("slots", SLOTS)]
        if list(b.get("targets", TARGETS)) != TARGETS or b_slots != [s[0] for s in SLOTS]:
            log(f"legacy bundle {p.name}: target/slot contract differs; blending skipped")
            return {}
        gr, n_gr = int(b.get("group", 3)), int(b.get("n_group", 3))
        variant = str(b.get("model_variant", "dinov2-small")).split("-")[-1]
        key = json.dumps({"img": int(b.get("img", 224)), "group": gr,
                          "slices": gr * n_gr, "crop_mm": 160.0, "band": [0.20, 0.80],
                          "rules": RULES_LEGACY, "slots": [s[0] for s in SLOTS]},
                         sort_keys=True)
        ms = [{"id": f"legacy-f{f.get('fold', k)}", "fold": f.get("fold", k),
               "state": f["state_dict"], "holdout": None, "weight": LEGACY_WEIGHT,
               "target_weight": [LEGACY_MEMBER_WEIGHT_BY_TARGET.get(t, 0.0)
                                 for t in TARGETS],
               "pixel_group": key,
               "config": {"unfreeze_last": 6,
                          "variant": "base" if variant == "base" else "small",
                          "pool": "cls_mean_focal", "prior": True}}
              for k, f in enumerate(folds)]
        if ms:
            active = sorted(set(LEGACY_MEMBER_WEIGHT_BY_TARGET.values()))
            log(f"legacy bundle {p.name}: {len(ms)} fold(s) join with "
                f"target-specific per-member weights {active}")
        return {key: ms} if ms else {}
    except Exception as exc:
        log(f"legacy bundle unusable ({type(exc).__name__}: {exc}); blending skipped")
        return {}


def _run_member(path, m, dev, Cte, Mte, idx, starts, jitter):
    """Load, verify and predict one member on one device. Returns (pred, timings)."""
    t0 = time.time()
    with BUILD_LOCK:
        if "state" in m:
            state, fp = m["state"], None
        else:
            ck = torch.load(Path(path) / m["file"], map_location="cpu",
                            weights_only=False)
            state, fp = ck["model"], ck.get("fingerprint")
        model = build_model(int(m["config"]["unfreeze_last"]),
                            variant=m["config"]["variant"],
                            pool=m["config"].get("pool", "cls_mean"),
                            prior=bool(m["config"].get("prior", False))).to(dev)
        model.load_state_dict(state)
        if fp is not None:
            check_fingerprint(model, dev, IMG, fp, tag=f"{m['id']}: ")
        else:
            log(f"  {m['id']}: no stored fingerprint (legacy bundle) -- "
                f"accepted at reduced weight")
    t_ready = time.time()
    jitter_seed = SEED + int(hashlib.sha256(str(m["id"]).encode()).hexdigest()[:8], 16)
    public_member = "state" not in m
    predicted = predict_member(
        model, Cte, Mte, idx, dev, IMG, starts=starts, jitter=jitter,
        jitter_seed=jitter_seed, return_public_frontier=public_member,
    )
    if public_member:
        p, public_p = predicted
    else:
        p, public_p = predicted, None
    t_done = time.time()
    del model, state
    gc.collect()
    if dev.type == "cuda":
        with torch.cuda.device(dev):
            torch.cuda.empty_cache()
    passes = len(starts) * (2 if jitter else 1)
    return p, public_p, (t_ready - t0, (t_done - t_ready) / max(passes, 1))


def _combine(per_member):
    """Target-wise weighted mean of per-member percentile ranks."""
    all_ids = sorted({s for m in per_member for s in m["ids"]})
    pos = {s: i for i, s in enumerate(all_ids)}
    acc = np.zeros((len(all_ids), len(TARGETS)), np.float64)
    tot = np.zeros(len(TARGETS), np.float64)
    for m in per_member:
        target_weight = m.get("target_weight")
        w = np.asarray(target_weight if target_weight is not None else
                       [float(m.get("weight", 1.0))] * len(TARGETS),
                       dtype=np.float64)
        if w.shape != (len(TARGETS),) or np.any(w < 0):
            raise ValueError(f"invalid target weights for {m.get('id')}: {w}")
        r = pd.DataFrame(m["pred"]).rank(pct=True).to_numpy()
        acc[[pos[s] for s in m["ids"]]] += r * w[None, :]
        tot += w
    if np.any(tot <= 0):
        raise ValueError(f"at least one target has no ensemble vote: {tot}")
    return all_ids, acc / tot[None, :]


def infer_from_package(path, dev=None):
    """Predict the test split from an attached package of trained members.

    Identical pixel contract to the reference implementation -- same caches, same
    windows, same fingerprints, same rank transform -- with executive changes only:

    1. A work queue over the devices: each GPU pops the next member when free (the
       reference ran one device and surrendered TTA windows, then members: 0.847).
    2. submission.csv is rewritten after every banked member, so a run killed at any
       point still submits the best partial ensemble instead of the 0.5 benchmark.
    3. A member that fails on one device is retried once on the other, then dropped;
       a dropped member costs one vote, never the run.
    4. An independently trained legacy bundle joins as four target-selective reduced-weight members.
    5. When the time estimate says the whole remaining ensemble fits with room to
       spare, each window gains one jittered TTA view (same family as training aug).
    """
    man = json.loads((Path(path) / "manifest.json").read_text())
    members = man["members"]
    log(f"weights package: {len(members)} member(s) from {path}; "
        f"{len(DEVS)} device(s)")

    test_df = pd.read_csv(ROOT / "test.csv")
    test_series = pd.read_csv(ROOT / "test_series.csv")
    plane_map = dict(zip(test_series["SeriesInstanceUID"],
                         test_series["Anatomical_Plane"]))
    hte = annotate(walk("test_series"))
    log(f"test header pass: {len(hte)} series")

    groups = {}
    for m in members:
        groups.setdefault(m["pixel_group"], []).append(m)
    # Legacy votes come last: if time binds after all, the weakest votes are the ones
    # surrendered, not the package's.
    groups.update(legacy_group_members())

    per_member, public_frontier_members = [], []
    est = {"fixed": None, "win": None}

    def bank(m, ids, pred, starts, jitter, public_pred=None):
        if float(np.std(pred)) < 1e-9:
            log(f"  {m['id']}: degenerate predictions; not banked")
            return
        with STATE_LOCK:
            per_member.append({"id": m["id"], "ids": ids, "pred": pred,
                               "weight": m.get("weight", 1.0),
                               "target_weight": m.get("target_weight"),
                               "holdout": m.get("holdout")})
            if public_pred is not None and len(starts) == len(starts_full):
                if float(np.std(public_pred)) < 1e-9:
                    raise WeightsError(f"{m['id']}: degenerate public-frontier prediction")
                public_frontier_members.append(
                    {"id": m["id"], "ids": ids, "pred": public_pred}
                )
            elif public_pred is not None:
                log(
                    f"  {m['id']}: public-frontier vote omitted because only "
                    f"{len(starts)} / {len(starts_full)} windows completed"
                )
            all_ids, acc = _combine(per_member)
            write_submission(acc, all_ids, test_df, "submission.csv")
            log(f"  banked {m['id']} fold {m.get('fold', '?')} "
                f"({len(starts)} window(s){', jitter' if jitter else ''}); "
                f"submission.csv = weighted rank mean of {len(per_member)} member(s)")

    for gi, (key, gm) in enumerate(groups.items(), 1):
        cfg = json.loads(key)
        adopt_config_globals(cfg)
        log(f"decode group {gi}/{len(groups)}: {cfg['img']}px x {cfg['slices']} slices, "
            f"crop {cfg['crop_mm']} mm -> {len(gm)} member(s)")
        st_te, Cte, Mte = build_cache(pick_slots(hte, plane_map), plane_map,
                                      lat_of(hte, "test "), f"test g{gi}")
        idx = np.arange(len(st_te))

        starts_full = window_starts(Cte.shape[2], GROUP)
        pending = sorted(gm, key=lambda m: -(m.get("holdout") or 0))
        left_after = sum(len(g) for j, (_, g) in enumerate(groups.items(), 1) if j > gi)

        def pop_next():
            """Next member, its window count, and whether jitter TTA is affordable.

            Windows are surrendered before members; jitter is granted only when the
            estimate says the whole remaining ensemble fits at double passes inside
            60% of the room. The question asked is whether ONE more member fits.
            """
            with STATE_LOCK:
                if not pending:
                    return None, None, False
                left = TIME_BUDGET - (time.time() - T0)
                remaining = len(pending) + left_after
                slots_left = -(-remaining // len(DEVS))       # ceil: concurrent slots
                starts, jit = starts_full, False
                if est["fixed"] is not None and est["win"] is not None:
                    afford = max(left * 0.9, 0.0)
                    room = afford / max(slots_left, 1)
                    if est["fixed"] + est["win"] > room:
                        log(f"  {left / 60:.0f} min left: surrendering "
                            f"{len(pending)} member(s); not one more fits")
                        pending.clear()
                        return None, None, False
                    jit = (est["fixed"] + 2 * len(starts_full) * est["win"]
                           <= room * 0.6)
                    per_win = est["win"] * (2 if jit else 1)
                    n_win = (int((room - est["fixed"]) / per_win)
                             if per_win > 0 else len(starts_full))
                    n_win = max(1, min(len(starts_full), n_win))
                    if n_win < len(starts_full):
                        mid = (len(starts_full) - n_win) // 2
                        starts = starts_full[mid:mid + n_win]
                return pending.pop(0), starts, jit

        def worker(dev):
            others = [d for d in DEVS if d is not dev]
            while True:
                m, starts, jit = pop_next()
                if m is None:
                    return
                for attempt, d in enumerate([dev] + others[:1]):
                    try:
                        p, public_p, (fs, ws) = _run_member(
                            path, m, d, Cte, Mte, idx, starts, jit
                        )
                        with STATE_LOCK:
                            est["fixed"], est["win"] = fs, ws
                        bank(m, st_te, p, starts, jit, public_p)
                        break
                    except Exception as exc:
                        log(f"  MEMBER {m['id']} failed on {d} "
                            f"({type(exc).__name__}: {exc}); "
                            + ("retrying on peer device" if attempt == 0 and others
                               else "dropped -- costs one vote, not the run"))
                        if d.type == "cuda":
                            with torch.cuda.device(d):
                                torch.cuda.empty_cache()

        threads = [threading.Thread(target=worker, args=(d,)) for d in DEVS]
        for t in threads:
            t.start()
        for t in threads:
            t.join()

        del Cte, Mte
        gc.collect()

    if not per_member:
        raise WeightsError("no member produced predictions; submission stays at 0.5")

    all_ids, acc = _combine(per_member)
    sub = write_submission(acc, all_ids, test_df, "submission.csv")
    log(f"final submission.csv = weighted rank mean of {len(per_member)} member(s); "
        f"{sub.shape}; nulls {int(sub[TARGETS].isna().sum().sum())}")
    if len(public_frontier_members) == len(members):
        frontier_ids, frontier_acc = _combine(public_frontier_members)
        frontier_sub = write_submission(
            frontier_acc, frontier_ids, test_df, "submission_public_0899.csv"
        )
        log(
            f"submission_public_0899.csv = exact no-jitter public-frontier rank mean "
            f"of {len(public_frontier_members)} member(s); {frontier_sub.shape}; "
            f"nulls {int(frontier_sub[TARGETS].isna().sum().sum())}"
        )
    else:
        log(
            f"public-frontier fallback not emitted: {len(public_frontier_members)} / "
            f"{len(members)} required public members completed"
        )
    return sub


def adopt_config_globals(cfg):
    """Point the pixel path at what one group of members was fitted on."""
    global IMG, CACHE_IMG, GROUP, CACHE_SLICES, N_GROUP, CROP_MM, SLICE_BAND, RULES
    CACHE_IMG = IMG = int(cfg["img"])
    GROUP = int(cfg["group"])
    CACHE_SLICES = int(cfg["slices"])
    N_GROUP = max(CACHE_SLICES // GROUP, 1)
    CROP_MM = float(cfg["crop_mm"])
    SLICE_BAND = tuple(float(x) for x in cfg["band"])
    # The four decisions that change what a slice is. A member fitted under one reading
    # and decoded under another gets pixels its weights never saw, with every shape
    # still agreeing, so an unrecognised name is refused rather than defaulted.
    rules = cfg.get("rules") or RULES_NATIVE
    unknown = {k: v for k, v in rules.items()
               if k not in RULES_NATIVE
               or v not in (RULES_NATIVE[k], RULES_LEGACY[k])}
    if unknown:
        raise WeightsError(f"the members record pixel rules this pipeline cannot "
                           f"reproduce: {unknown}")
    RULES = {**RULES_NATIVE, **rules}
    if [s[0] for s in SLOTS] != list(cfg["slots"]):
        raise WeightsError(
            f"the members were fitted on slots {cfg['slots']} and this pipeline defines "
            f"{[s[0] for s in SLOTS]}; a weight would be read against the wrong slot")


In [19]:
def take_group(cache_rows, g):
    """Slice GROUP consecutive channels out of the cached slices."""
    return cache_rows[:, :, g * GROUP:(g + 1) * GROUP]


def augment(imgs, generator=None):
    """A small rigid jitter and an intensity scale, applied to a whole bag at once.

    Neither flip is available here, and for different reasons. A horizontal flip would
    reintroduce the nuisance axis that the laterality normalisation removed - it would
    undo, once per batch, what the header pass was run to establish.

    A vertical flip is not a nuisance axis at all. A knee is acquired in a canonical
    orientation, and no study in this corpus looks like its own vertical mirror. An
    augmentation is meant to cover directions along which the label does not change; this
    one moves the input off the distribution the encoder will be asked about, which is a
    different thing. Where a finding sits in the frame is also information rather than
    noise - a Baker cyst is identified by lying in the popliteal fossa, not by its
    appearance alone.

    What is left is jitter that no label depends on: a few degrees of rotation, a few
    per cent of scale and translation. That still prevents memorising the exact framing,
    which is what an augmentation is for, while leaving the anatomy where it was.
    """
    # A bag arrives as [study, slot, GROUP, IMG, IMG]: five axes, not four. The warp is
    # a 2-D operation, so the two leading axes are folded together and restored after -
    # every slot image is an independent acquisition and gets its own jitter.
    lead = imgs.shape[:-3]
    x = imgs.reshape(-1, *imgs.shape[-3:]).float()
    n, dev = x.shape[0], x.device

    rot = (torch.rand(n, device=dev, generator=generator) - 0.5) * 2 * (AUG_ROT_DEG * np.pi / 180)
    # Zoom in only. `border` padding repeats the edge row outward, and the edge of this
    # crop is where the popliteal fossa sits; zooming out would fabricate tissue exactly
    # where a Baker cyst is looked for.
    sc = 1.0 + torch.rand(n, device=dev, generator=generator) * AUG_SCALE
    tx = (torch.rand(n, device=dev, generator=generator) - 0.5) * 2 * AUG_SHIFT
    ty = (torch.rand(n, device=dev, generator=generator) - 0.5) * 2 * AUG_SHIFT
    cos, sin = torch.cos(rot) / sc, torch.sin(rot) / sc
    theta = torch.zeros(n, 2, 3, device=dev, dtype=torch.float32)
    theta[:, 0, 0], theta[:, 0, 1], theta[:, 0, 2] = cos, -sin, tx
    theta[:, 1, 0], theta[:, 1, 1], theta[:, 1, 2] = sin, cos, ty
    grid = F.affine_grid(theta, x.shape, align_corners=False)
    x = F.grid_sample(x, grid, mode="bilinear", padding_mode="border", align_corners=False)

    scale = 1.0 + (torch.rand(n, 1, 1, 1, device=dev, generator=generator) - 0.5) * 2 * AUG_INTENSITY
    x = (x * scale).clamp(0, 255)
    return x.reshape(*lead, *x.shape[-3:]).to(imgs.dtype)


@torch.no_grad()
def predict(model, cache, mask, idx, dev, img_size=None):
    """Average the logits over the groups of each slot.

    Training sees one group at a time, which acts as augmentation along the stack;
    inference averages over all of them, so the prediction does not depend on which
    group a single draw happened to pick. Where the cache holds one group per slot the
    two coincide.
    """
    model.eval()
    out = []
    for b in range(0, len(idx), EVAL_BATCH):
        sel = idx[b:b + EVAL_BATCH]
        m = torch.from_numpy(mask[sel]).to(dev)
        acc = None
        for g in range(N_GROUP):
            # Gathered a group at a time rather than whole and then sliced. The two are
            # the same pixels, but taking the whole of a study out of the cache allocates
            # every slice it holds - most of which this pass will not look at until a
            # later iteration, by which time they have been fetched again. Measured over
            # a cache of twelve slices, the difference between the two is the difference
            # between the step being bound by memory and being bound by the encoder.
            rows = torch.from_numpy(np.ascontiguousarray(
                cache[sel, :, g * GROUP:(g + 1) * GROUP])).to(dev)
            with torch.autocast("cuda", enabled=dev.type == "cuda"):
                z = model(rows, m, img_size).float()
            acc = z if acc is None else acc + z
        out.append(torch.sigmoid(acc / N_GROUP).cpu().numpy())
    return np.concatenate(out) if out else np.zeros((0, len(TARGETS)), np.float32)


def macro_auc(y, p):
    from sklearn.metrics import roc_auc_score
    return float(np.nanmean([roc_auc_score(y[:, j], p[:, j])
                             if len(set(y[:, j])) > 1 else np.nan
                             for j in range(y.shape[1])]))


In [22]:
def write_submission(pred, studies, test_df, path):
    """Write one submission file from a prediction matrix.

    Predictions are converted to per-column ranks first: the metric reads only order, so
    ranks discard nothing, and they make files from different configurations directly
    comparable and safe to average.
    """
    sub = pd.DataFrame(pd.DataFrame(pred).rank(pct=True).values, columns=TARGETS)
    sub.insert(0, "StudyInstanceUID", studies)
    sub = test_df[["StudyInstanceUID"]].merge(sub, on="StudyInstanceUID", how="left")
    sub[TARGETS] = sub[TARGETS].fillna(0.5)
    sub.to_csv(path, index=False)
    return sub


def write_benchmark_submission():
    """Write the 0.5 benchmark file immediately.

    A submission that never writes scores nothing at all, which is strictly worse than
    scoring badly. The try/except around main() covers exceptions, but a kill for memory
    is a SIGKILL and never reaches it. So a valid file exists from the first second and
    is overwritten only once real predictions are ready.
    """
    t = pd.read_csv(ROOT / "test.csv")
    for c in TARGETS:
        t[c] = 0.5
    t.to_csv("submission.csv", index=False)


def _v37_validate_submission(path, test_df, tag):
    """Read one attached prediction file only after its full contract passes."""
    path = Path(path)
    frame = pd.read_csv(path)
    expected = ["StudyInstanceUID"] + TARGETS
    if list(frame.columns) != expected:
        raise ValueError(f"{tag}: columns differ from the competition contract")
    if len(frame) != len(test_df) or not frame["StudyInstanceUID"].is_unique:
        raise ValueError(f"{tag}: row count or StudyInstanceUID uniqueness failed")
    if set(frame["StudyInstanceUID"].astype(str)) != set(test_df["StudyInstanceUID"].astype(str)):
        raise ValueError(f"{tag}: StudyInstanceUID set differs from test.csv")
    values = frame[TARGETS].to_numpy(np.float64)
    if not np.isfinite(values).all():
        raise ValueError(f"{tag}: non-finite prediction")
    return test_df[["StudyInstanceUID"]].merge(frame, on="StudyInstanceUID", how="left")


def _v37_find_yash_submission():
    """Find the output mounted from the exact public Yash notebook source."""
    candidates = []
    local = globals().get("YASH_LOCAL_SOURCE_DIR")
    if local:
        candidates.append(Path(local) / "submission.csv")
    root = Path("/kaggle/input")
    candidates.append(root / "rsna-knee-infer-v1" / "submission.csv")
    if root.is_dir():
        candidates.extend(meta.parent / "submission.csv"
                          for meta in root.glob("**/infer_meta.json"))
    seen = set()
    for path in candidates:
        key = str(path)
        if key in seen or not path.is_file():
            continue
        seen.add(key)
        meta_path = path.with_name("infer_meta.json")
        if meta_path.is_file():
            meta = json.loads(meta_path.read_text())
            if int(meta.get("errors", -1)) != 0:
                raise ValueError(f"Yash source reports {meta.get('errors')} inference errors")
        return path
    raise FileNotFoundError("the attached yashbishnoi98/rsna-knee-infer-v1 output is absent")


def run_yash_public_ensemble():
    """Bank the public image specialist and a conservative rank ensemble.

    The Yash source is an independently trained EfficientNet-B3, five-fold,
    plane-aware image family. The local public fallback is the independently
    published twenty-member DINO family. A 0.55/0.45 Borda blend keeps Yash as
    the stronger voter but lets concordant DINO evidence resolve close orderings.
    The exact public source and the pre-blend native primary are both retained.
    """
    import shutil

    test_df = pd.read_csv(ROOT / "test.csv")
    native_path = Path("submission.csv")
    public_path = Path("submission_public_0899.csv")
    native = _v37_validate_submission(native_path, test_df, "native V36")
    public = _v37_validate_submission(public_path, test_df, "public DINO family")
    yash_path = _v37_find_yash_submission()
    yash = _v37_validate_submission(yash_path, test_df, "Yash public image family")
    meta_path = yash_path.with_name("infer_meta.json")
    if meta_path.is_file():
        meta = json.loads(meta_path.read_text())
        if int(meta.get("studies", -1)) != len(test_df):
            raise ValueError("Yash source study count differs from test.csv")

    shutil.copyfile(native_path, "submission_native_v36.csv")
    shutil.copyfile(yash_path, "submission_yash_reference.csv")
    yr = yash[TARGETS].rank(pct=True).to_numpy(np.float64)
    dr = public[TARGETS].rank(pct=True).to_numpy(np.float64)
    blend = 0.55 * yr + 0.45 * dr
    result = test_df[["StudyInstanceUID"]].copy()
    result[TARGETS] = blend
    if result.shape != yash.shape or not np.isfinite(result[TARGETS].to_numpy()).all():
        raise AssertionError("invalid Yash/DINO rank blend")
    candidate_path = Path("submission_yash_dino_rankblend.csv")
    result.to_csv(candidate_path, index=False)
    reread = _v37_validate_submission(candidate_path, test_df, "Yash/DINO rank blend")
    changed = sum(
        tuple(reread[target].rank(method="first")) !=
        tuple(yash[target].rank(method="first"))
        for target in TARGETS
    )
    if changed == 0:
        raise AssertionError("Yash/DINO blend is rank-identical to its Yash parent")
    temp_path = Path("submission_v37_yash_dino.tmp.csv")
    reread.to_csv(temp_path, index=False)
    temp_path.replace(native_path)
    log(f"Yash public family banked; V37 primary = 0.55 Yash / 0.45 public DINO "
        f"rank blend ({changed} target orderings differ from Yash); exact Yash and "
        "native V36 outputs retained")
    return True


def main():
    write_benchmark_submission()

    # Weights, if any were attached; otherwise the run learns its own below. Both paths
    # are kept because the second is what makes this notebook readable on its own - a
    # fork with nothing attached still trains and still scores - and because the first
    # cannot be checked by anyone who does not have the package.
    pkg = find_weights()
    if pkg is not None:
        dev = DEVS[0]
        infer_from_package(pkg, dev)

        # The exact no-jitter target-pooling recipe independently completed at 0.899 in
        # the public frontier.  Earlier versions retained it as a secondary artifact and
        # then promoted less certain specialists.  Make the evidence-backed arm primary
        # only after the complete hidden UID/schema/finite-value contract passes.
        try:
            test_df = pd.read_csv(ROOT / "test.csv")
            native_path = Path("submission.csv")
            public_path = Path("submission_public_0899.csv")
            native = _v37_validate_submission(native_path, test_df, "native 24-member")
            public = _v37_validate_submission(public_path, test_df, "public DINO frontier")
            native.to_csv("submission_native_v38.csv", index=False)
            public.to_csv(native_path, index=False)
            promoted = _v37_validate_submission(native_path, test_df, "V40 primary")
            if not promoted.equals(public):
                raise AssertionError("V40 serialization differs from validated public frontier")
            log("V40 primary = exact no-jitter public-frontier target pooling; "
                "native 24-member output retained")
        except Exception as public_frontier_error:
            log(f"public-frontier promotion skipped safely: {public_frontier_error}")
            traceback.print_exc()
        log("done")
        return

    # Settle where the labels come from before anything expensive runs. The check costs
    # one CSV header read; discovering the same problem after the cache is built would
    # cost the whole decode pass, and discovering it never would cost the run.
    read_labels(pd.read_csv(ROOT / "train.csv", usecols=["StudyInstanceUID", "Report"]))

    test_df = pd.read_csv(ROOT / "test.csv")
    test_series = pd.read_csv(ROOT / "test_series.csv")
    train_df = pd.read_csv(ROOT / "train.csv")
    train_series = pd.read_csv(ROOT / "train_series.csv")
    log(f"train {train_df.shape} test {test_df.shape}")

    both = pd.concat([train_series, test_series])
    plane_map = dict(zip(both["SeriesInstanceUID"], both["Anatomical_Plane"]))

    log("header pass: test")
    hte = annotate(walk("test_series"))
    log(f"  {len(hte)} test series")
    log("header pass: train")
    htr = annotate(walk("train_series"))
    log(f"  {len(htr)} train series")

    slots_te, slots_tr = pick_slots(hte, plane_map), pick_slots(htr, plane_map)
    cov = pd.Series([len(v) for v in slots_tr.values()]).describe()
    log(f"train slots per study: mean {cov['mean']:.2f} min {cov['min']:.0f} "
        f"max {cov['max']:.0f}")

    st_tr, Ctr, Mtr = build_cache(slots_tr, plane_map, lat_of(htr, "train "), "train")
    st_te, Cte, Mte = build_cache(slots_te, plane_map, lat_of(hte, "test "), "test")

    # ---- targets ---------------------------------------------------------- #
    t_lab = time.time()
    lab = read_labels(train_df)
    log(f"derived labels for {len(lab)} studies in {time.time() - t_lab:.1f}s")

    gold = train_df.set_index("StudyInstanceUID")[TARGETS]
    gold = gold[gold.notna().all(axis=1)]

    Y = np.zeros((len(st_tr), len(TARGETS)), np.float32)
    W = np.zeros_like(Y)
    for i, st in enumerate(st_tr):
        if st in gold.index:
            Y[i], W[i] = gold.loc[st].values, 3.0
        elif st in lab.index:
            r = lab.loc[st]
            Y[i] = r[TARGETS].values
            W[i] = 0.25 + 0.75 * r[[t + "__conf" for t in TARGETS]].values
    keep = np.where(W.sum(1) > 0)[0]
    log(f"supervised {len(keep)} of {len(st_tr)} studies (annotated {len(gold)})")

    # Grouped on report text: some reports are byte-identical across studies and yield
    # one target vector for all of them, so splitting such a group scores the model on a
    # target whose source it has already trained on.
    import hashlib
    rep = train_df.set_index("StudyInstanceUID")["Report"].fillna("")
    grp = np.array([int(hashlib.md5(rep.get(s, s).encode()).hexdigest()[:8], 16) % 5
                    for s in st_tr])
    va = np.array([i for i in keep if grp[i] == 0])
    tr = np.array([i for i in keep if grp[i] != 0])
    if len(va) == 0 or len(tr) < BATCH_STUDIES:
        cut = max(1, len(keep) // 5)
        va, tr = keep[:cut], keep[cut:]
    log(f"train {len(tr)} / holdout {len(va)} studies")

    # The annotated studies stay in training - they are the highest-quality labels in
    # the corpus and there are too few to discard - so the honest annotation check uses
    # only the ones that fell in the holdout. Evaluating on the rest would be scoring the
    # model against examples it was trained on, at triple weight, with the true answer.
    gpos = {s: i for i, s in enumerate(st_tr)}
    va_set = set(va.tolist())
    gi = np.array([gpos[s] for s in gold.index if s in gpos and gpos[s] in va_set])
    gold_y = gold.loc[[st_tr[i] for i in gi]].values.astype(int) if len(gi) else None
    yv = (Y[va] > 0.5).astype(int)
    log(f"annotation check: {len(gi)} of {len(gold)} annotated studies are in the holdout")

    # ---- fine-tune -------------------------------------------------------- #
    dev = DEVS[0]
    results, test_preds = {}, {}

    for cfg in RUNS:
        pitch = CROP_MM / cfg["img"]
        log(f"=== {cfg['name']}: {cfg['img']} px, {pitch:.3f} mm/pixel, "
            f"{pitch * 14:.2f} mm per patch token ===")
        torch.manual_seed(SEED)
        model = build_model(UNFREEZE_LAST).to(dev)
        opt = torch.optim.AdamW([
            {"params": [p for p in model.backbone.parameters() if p.requires_grad],
             "lr": LR_BACKBONE},
            {"params": model.head.parameters(), "lr": LR_HEAD},
        ], weight_decay=WEIGHT_DECAY)
        steps = max(EPOCHS * (len(tr) // BATCH_STUDIES), 1)
        sched = torch.optim.lr_scheduler.OneCycleLR(
            opt, max_lr=[LR_BACKBONE, LR_HEAD], total_steps=steps, pct_start=0.15)
        scaler = torch.amp.GradScaler("cuda", enabled=dev.type == "cuda")

        best, best_state, best_annot = -1.0, None, float("nan")
        for ep in range(EPOCHS):
            model.train()
            perm = np.random.permutation(tr)
            tot, nstep = 0.0, 0
            for b in range(0, len(perm) - BATCH_STUDIES + 1, BATCH_STUDIES):
                sel = perm[b:b + BATCH_STUDIES]
                rows = torch.from_numpy(Ctr[sel]).to(dev)
                g = int(torch.randint(N_GROUP, (1,)).item())
                imgs = augment(take_group(rows, g))
                m = torch.from_numpy(Mtr[sel]).to(dev)
                y = torch.from_numpy(Y[sel]).to(dev)
                w = torch.from_numpy(W[sel]).to(dev)
                with torch.autocast("cuda", enabled=dev.type == "cuda"):
                    loss = (F.binary_cross_entropy_with_logits(
                        model(imgs, m, cfg["img"]), y, reduction="none") * w).mean()
                opt.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.step(opt)
                scaler.update()
                sched.step()
                tot += loss.item()
                nstep += 1

            pv = predict(model, Ctr, Mtr, va, dev, cfg["img"])
            d = macro_auc(yv, pv)
            g_auc = float("nan")
            if gold_y is not None and len(gi):
                g_auc = macro_auc(gold_y, predict(model, Ctr, Mtr, gi, dev, cfg["img"]))
            log(f"  epoch {ep + 1}/{EPOCHS}  loss {tot / max(nstep, 1):.4f}"
                f"  holdout {d:.4f}  annot(n={len(gi)}) {g_auc:.4f}")

            # Selection reads the holdout alone. The annotation check is reported because
            # it measures something different - agreement with a reading of the images
            # rather than of the reports - but only a handful of annotated studies land
            # in any one holdout, so its sampling error dwarfs the differences between
            # epochs and it cannot arbitrate between them.
            if d > best:
                best, best_annot = d, g_auc
                best_state = {k: v.detach().cpu().clone()
                              for k, v in model.state_dict().items()}
            if time.time() - T0 > TIME_BUDGET:
                log("  time budget reached")
                break

        if best_state is not None:
            model.load_state_dict(best_state)
        results[cfg["name"]] = (best, best_annot)
        test_preds[cfg["name"]] = predict(model, Cte, Mte, np.arange(len(st_te)), dev,
                                          cfg["img"])
        log(f"  {cfg['name']}: best holdout {best:.4f} (annot {best_annot:.4f})")
        del model, opt, sched, scaler, best_state
        gc.collect()
        if dev.type == "cuda":
            torch.cuda.empty_cache()

    log("---- summary ----")
    for n, (d, g_auc) in results.items():
        log(f"  {n:12s} holdout {d:.4f}   annot {g_auc:.4f}")
    pick = max(results, key=lambda k: results[k][0])
    log(f"best on the holdout: {pick} ({results[pick][0]:.4f})")


    # ---- write every candidate -------------------------------------------- #
    # One file per configuration, plus the holdout's choice as `submission.csv`. A run
    # costs a full decode of the corpus whichever configuration wins, so keeping every
    # arm makes a later change of configuration free rather than another full run.
    for name, pred in test_preds.items():
        sub = write_submission(pred, st_te, test_df, f"submission_{name}.csv")
        log(f"  submission_{name}.csv {sub.shape}; "
            f"nulls {int(sub[TARGETS].isna().sum().sum())}")

    ens = np.mean([pd.DataFrame(p).rank(pct=True).values for p in test_preds.values()],
                  axis=0)
    write_submission(ens, st_te, test_df, "submission_rankmean.csv")
    log(f"  submission_rankmean.csv (rank mean of {len(test_preds)})")

    sub = write_submission(test_preds[pick], st_te, test_df, "submission.csv")
    log(f"submission.csv = {pick}; {sub.shape}; "
        f"nulls {int(sub[TARGETS].isna().sum().sum())}")
    print(sub.head().to_string())


In [23]:
try:
    main()
except LabelSourceError:
    # Deliberately not absorbed: see LabelSourceError. A run that trained on the
    # wrong labels would finish and write a submission worth submitting by mistake.
    traceback.print_exc()
    raise
except Exception:
    traceback.print_exc()
    # A submission that fails to write scores nothing at all, so fall back to the
    # benchmark file rather than dying.
    t = pd.read_csv(find_root() / "test.csv")
    for c in TARGETS:
        t[c] = 0.5
    t.to_csv("submission.csv", index=False)
    print("wrote fallback submission.csv")
log("done")


In [ ]:
# E9: independent RadImageNet ResNet-50 arm for the verified E2 parent.
#
# Adapted 2026-08-11 from the V52 cell in the public Kaggle competition notebook
# prvsiyan/rsna-knee-read-the-report-then-the-knee (latest source SHA-256
# b54aa529f38dc6f594478e7975d86459ddbff898453a2c2633f4c3be4b909e61).
# Kaggle's public-code rule deems public Competition Code open-source; Meta Kaggle
# documents public notebooks under Apache-2.0. Changes here remove the unavailable B3
# arm, pin the public E2 OOF bundle, add per-target/no-regression gates, and preserve E2
# byte-for-byte on every failure. This module is appended as the final notebook cell;
# its imports and DICOM helpers are supplied by the parent notebook.
SLOTS = [
    ("SAG_FS", "Sagittal", None, True),
    ("COR_FS", "Coronal", None, True),
    ("AX_FS", "Axial", None, True),
]
N_SLOT = len(SLOTS)
CACHE_SLICES = 8
TIME_BUDGET = 8.72 * 3600
IMG = CACHE_IMG = 224
# Match the released RadImageNet model's full-frame pretraining. Setting a crop
# larger than every acquisition disables the optional physical crop in read_slot.
CROP_MM = 10_000.0
SLICE_BAND = (0.12, 0.88)
# The audited public OOF was produced after the notebook's legacy member group, which
# left this process-global pixel contract active. Pin it instead of inheriting whichever
# E2 group happened to run last. This uses public preprocessing code only; no legacy
# checkpoint or unknown-license asset is attached.
RULES = dict(RULES_LEGACY)
TOKEN_DIM = 2048             # official ResNet-50 global-average feature
HEAD_DIM = 512
PINNED_HEADS_SHA256 = "0f465649799ecfbccaac1767844639e7ced44e1bc9babde6e4bac7c5d9b89eaa"
PINNED_REMOTE_AUDIT_SHA256 = "267f948078710d3ca8a6f0de4ce0a5e75e850e1f08d36451549a137d878a6fe8"
PINNED_PUBLIC_DIAGNOSTIC_SHA256 = "0f2f82fb40f0570d6766f73b0d7f51489df6d0faa8fd6e4c0f45bcd6c4c7b283"
PINNED_E9B_CONTRACT_SHA256 = "6777c0a0ba7dd044752fac948752dc39e9ca35b3c280fe74ee6d27a5865d87e7"
# E10 repairs E9's censored search: its alpha grid stopped at 0.25 and four of five outer
# folds selected that ceiling, so the deployed 0.20 was a boundary artifact rather than an
# optimum. The ladder, the two-source per-target gains and every deployable weight map live
# in the hash-pinned contract; this cell recomputes the remote half in-kernel before use.
# The contract also carries the held-out form of E10's own weight choice: selecting the rung
# on four grouped folds and scoring the fifth picks 0.60 (public) and 0.70 (v15) in all five
# outer folds, and never picks 0.20. So every deployable rung at or below 0.35 is below what
# honest selection would choose, which is the answer to "you tuned on the 58 gold rows".
PINNED_E10_CONTRACT_SHA256 = "219c91f40905181c222e2966b3fed01a96570ddfb64862357d5fd6cad500cd45"
E10_CONFIG = "per_target_nested"
E10_PRESERVED_TARGETS = ["Baker's", "Fracture"]

# E11 trains a third arm whose diversity is in the pixels rather than in the weights. The
# existing arm reads three fat-suppressed slots at full frame; every one of E2's twenty
# members reads one DINOv2 recipe. Nothing in the portfolio has yet looked at a
# non-fat-suppressed series, where meniscal and ligament morphology is conventionally read,
# and nothing has given RadImageNet a physically normalised field of view. E11 changes both:
# three non-suppressed slots plus one suppressed anchor, cropped to 130 mm, which the parent
# notebook establishes is below the acquired field of view of 99.6% of series while still
# containing the joint. It is a training mode only; it never touches the submission.
ARM_MODE = "e10"
E11_SLOTS = [
    ("SAG_NOFS", "Sagittal", None, False),
    ("COR_NOFS", "Coronal", None, False),
    ("AX_NOFS", "Axial", None, False),
    ("SAG_FS", "Sagittal", None, True),
]
E11_CROP_MM = 130.0
E11_CACHE_SLICES = 8
E11_IMG = 224
# Availability of non-suppressed series per plane is unmeasured, so a fill floor rather than
# the parent's 90% rule: below this the run has found something structurally wrong, above it
# an empty slot is simply masked out of the token set like any other absent series.
E11_MIN_FILL = 0.45


def _v52_as_bool(value):
    if pd.isna(value):
        return None
    text = str(value).strip().upper()
    if text in {"1", "TRUE", "T", "YES", "Y"}:
        return True
    if text in {"0", "FALSE", "F", "NO", "N"}:
        return False
    try:
        number = float(text)
        return True if number == 1 else False if number == 0 else None
    except Exception:
        return None


def audit_official_sequence_metadata(inferred, official):
    """Audit metadata agreement without changing the checkpoint pixel contract."""
    needed = {"SeriesInstanceUID", "Fluid_Sensitive", "Fat_Suppression"}
    if inferred.empty or official.empty or not needed.issubset(official.columns):
        return
    inferred_flags = inferred[["SeriesInstanceUID", "fluid", "fatsat"]].copy()
    official_flags = official[
        ["SeriesInstanceUID", "Fluid_Sensitive", "Fat_Suppression"]
    ].copy()
    official_flags["official_fluid"] = official_flags["Fluid_Sensitive"].map(
        _v52_as_bool
    )
    official_flags["official_fatsat"] = official_flags["Fat_Suppression"].map(
        _v52_as_bool
    )
    merged = inferred_flags.merge(
        official_flags[["SeriesInstanceUID", "official_fluid", "official_fatsat"]],
        on="SeriesInstanceUID",
        how="inner",
    )
    for inferred_col, official_col, name in [
        ("fluid", "official_fluid", "Fluid_Sensitive"),
        ("fatsat", "official_fatsat", "Fat_Suppression"),
    ]:
        valid = merged[official_col].notna() & merged[inferred_col].notna()
        if valid.any():
            agreement = (
                merged.loc[valid, inferred_col].astype(bool).to_numpy()
                == merged.loc[valid, official_col].astype(bool).to_numpy()
            ).mean()
            log(
                f"V52 metadata audit {name}: {agreement:.1%} agreement "
                f"on {int(valid.sum())} series"
            )


def find_input_file(name):
    for root, dirs, files in os.walk("/kaggle/input"):
        dirs[:] = [d for d in dirs if d not in ("train_series", "test_series")]
        if name in files:
            return Path(root) / name
    raise FileNotFoundError(name)


def find_input_dir(name):
    for root, dirs, files in os.walk("/kaggle/input"):
        if Path(root).name == name:
            return Path(root)
    raise FileNotFoundError(name)


def make_targets(train):
    """Three independent public report teachers; image-read gold always wins."""
    uid = "StudyInstanceUID"
    sources = [
        pd.read_csv(find_input_file("report_labels_v2.csv")),
        pd.read_csv(find_input_file("llm_labels_v2.csv")),
        pd.read_csv(find_input_file("labels_llm_gpt56sol.csv")),
    ]
    cube = []
    for frame in sources:
        if frame[uid].duplicated().any():
            raise ValueError("duplicate study in report-label source")
        aligned = train[[uid]].merge(frame[[uid] + TARGETS], on=uid, how="left")
        cube.append(aligned[TARGETS].to_numpy(float))
    cube = np.stack(cube)
    available = np.isfinite(cube).sum(0)
    if np.any(available < 2):
        raise ValueError("fewer than two report teachers for a study/target")
    y = np.nanmean(cube, axis=0).astype(np.float32)
    disagreement = np.nanmean(np.abs(cube - y[None]), axis=0)
    agreement = np.clip(1.0 - 2.0 * disagreement, 0, 1)
    certainty = np.clip(2.0 * np.abs(y - .5), 0, 1)
    w = (.15 + .85 * (.65 * agreement + .35 * certainty)).astype(np.float32)
    gold = train[TARGETS].notna().all(axis=1).to_numpy()
    y[gold] = train.loc[gold, TARGETS].to_numpy(np.float32)
    w[gold] = 3.0
    return y, w, gold


def report_groups(train):
    report = (train.Report.fillna("").astype(str).str.lower()
              .str.replace(r"\s+", " ", regex=True).str.strip())
    return np.array([hashlib.sha256(x.encode()).hexdigest()[:24] for x in report])


def _v52_sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(8 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def load_radimagenet(device):
    """Strictly load the official RadImageNet ResNet-50 PyTorch checkpoint."""
    from torchvision.models import resnet50

    checkpoint = find_input_file("ResNet50.pt")
    expected_checkpoint = "08629f7e7bd3e29b8ee9522ca3f65ce4d010a7ddf74f0ea3c7e3f3d0bbab0734"
    observed_checkpoint = _v52_sha256(checkpoint)
    if observed_checkpoint != expected_checkpoint:
        raise RuntimeError(f"RadImageNet checkpoint drift: {observed_checkpoint}")

    class RadImageNetEncoder(nn.Module):
        def __init__(self):
            super().__init__()
            self.backbone = nn.Sequential(
                *list(resnet50(weights=None).children())[:-2]
            )

        def forward(self, image):
            return self.backbone(image).mean(dim=(2, 3))

    model = RadImageNetEncoder()
    state = torch.load(checkpoint, map_location="cpu", weights_only=True)
    if not state or not all(str(key).startswith("backbone.") for key in state):
        raise RuntimeError("unexpected RadImageNet state-dict namespace")
    model.load_state_dict(state, strict=True)
    parameter_count = sum(parameter.numel() for parameter in model.parameters())
    if parameter_count != 23_508_032:
        raise RuntimeError(f"unexpected RadImageNet parameter count {parameter_count}")
    model.eval().to(device)
    for parameter in model.parameters():
        parameter.requires_grad_(False)
    gpu_count = torch.cuda.device_count() if device.type == "cuda" else 0
    if gpu_count > 1:
        model = nn.DataParallel(model, device_ids=list(range(gpu_count)))
    log(
        f"RadImageNet strict load: {parameter_count:,} params; "
        f"inference GPUs={max(1, gpu_count)}"
    )
    return model


@torch.inference_mode()
def encode_radimagenet(cache, slot_mask, device):
    """Encode acquired slices with the official [-1, 1] RadImageNet contract."""
    n, slots, slices, h, w = cache.shape
    features = np.zeros((n, slots * slices, TOKEN_DIM), np.float16)
    token_mask = np.repeat(slot_mask[:, :, None], slices, axis=2).reshape(n, -1)
    valid = np.flatnonzero(token_mask.reshape(-1) > 0)
    flat = cache.reshape(-1, h, w)
    model = load_radimagenet(device)
    if device.type == "cuda":
        batch = 192 if torch.cuda.device_count() > 1 else 96
    else:
        batch = 8
    for b0 in range(0, len(valid), batch):
        ix = valid[b0:b0 + batch]
        x = torch.from_numpy(flat[ix]).to(device).float().div_(127.5).sub_(1.0)
        x = x.unsqueeze(1).expand(-1, 3, -1, -1).contiguous()
        with torch.autocast("cuda", enabled=device.type == "cuda"):
            feat = model(x)
        if feat.shape[1:] != (TOKEN_DIM,):
            raise RuntimeError(f"unexpected RadImageNet feature shape {tuple(feat.shape)}")
        features.reshape(-1, TOKEN_DIM)[ix] = (
            feat.float().cpu().numpy().astype(np.float16)
        )
        if b0 % (batch * 100) == 0:
            log(f"RadImageNet encoded {b0}/{len(valid)} acquired slices")
    del model
    if device.type == "cuda":
        torch.cuda.empty_cache()
    return features, token_mask.astype(np.float32)


class FoundationQueryHead(nn.Module):
    def __init__(self):
        super().__init__()
        self.project = nn.Sequential(nn.LayerNorm(TOKEN_DIM),
                                     nn.Linear(TOKEN_DIM, HEAD_DIM), nn.GELU())
        self.plane = nn.Parameter(torch.randn(N_SLOT, HEAD_DIM) * .01)
        self.position = nn.Parameter(torch.randn(CACHE_SLICES, HEAD_DIM) * .01)
        self.query = nn.Parameter(torch.randn(len(TARGETS), HEAD_DIM) * .02)
        self.attn = nn.MultiheadAttention(HEAD_DIM, 8, dropout=.10, batch_first=True)
        self.fuse = nn.Sequential(
            nn.LayerNorm(HEAD_DIM * 4), nn.Linear(HEAD_DIM * 4, HEAD_DIM),
            nn.GELU(), nn.Dropout(.15),
        )
        self.weight = nn.Parameter(torch.randn(len(TARGETS), HEAD_DIM) * .02)
        self.bias = nn.Parameter(torch.zeros(len(TARGETS)))

    def forward(self, feature, mask):
        token = self.project(feature.float())
        token = token.view(len(token), N_SLOT, CACHE_SLICES, HEAD_DIM)
        token = token + self.plane[None, :, None] + self.position[None, None]
        token = token.flatten(1, 2)
        key_padding = mask <= 0
        # No study should be empty, but keep MHA numerically defined if one is.
        all_empty = key_padding.all(1)
        if all_empty.any():
            key_padding = key_padding.clone()
            key_padding[all_empty, 0] = False
        query = self.query.unsqueeze(0).expand(len(token), -1, -1)
        attended = query + self.attn(query, token, token,
                                     key_padding_mask=key_padding,
                                     need_weights=False)[0]
        denom = mask.sum(1, keepdim=True).clamp_min(1).unsqueeze(-1)
        mean = (token * mask.unsqueeze(-1)).sum(1, keepdims=True) / denom
        mean = mean.expand(-1, len(TARGETS), -1)
        fused = self.fuse(torch.cat(
            [attended, mean, torch.abs(attended - mean), attended * mean], -1))
        return (fused * self.weight.unsqueeze(0)).sum(-1) + self.bias


def macro_auc(y, pred):
    from sklearn.metrics import roc_auc_score
    hard = (np.asarray(y) >= .5).astype(np.uint8)
    values = [roc_auc_score(hard[:, j], pred[:, j])
              for j in range(hard.shape[1]) if np.unique(hard[:, j]).size == 2]
    return float(np.mean(values))


def _v52_target_auc(y, pred):
    from sklearn.metrics import roc_auc_score
    hard = (np.asarray(y) >= .5).astype(np.uint8)
    return {
        target: float(roc_auc_score(hard[:, index], pred[:, index]))
        for index, target in enumerate(TARGETS)
    }


@torch.inference_mode()
def predict_head(model, features, masks, indices, device, batch=64):
    model.eval()
    pred = []
    for b0 in range(0, len(indices), batch):
        ix = indices[b0:b0 + batch]
        x = torch.from_numpy(features[ix]).to(device)
        m = torch.from_numpy(masks[ix]).to(device)
        with torch.autocast("cuda", enabled=device.type == "cuda"):
            pred.append(torch.sigmoid(model(x, m)).float().cpu())
    return torch.cat(pred).numpy()


def train_fold(features, masks, y, weights, train_idx, val_idx, fold, device):
    from torch.utils.data import DataLoader, Dataset
    class Rows(Dataset):
        def __init__(self, indices): self.indices = np.asarray(indices)
        def __len__(self): return len(self.indices)
        def __getitem__(self, k):
            i = self.indices[k]
            return features[i], masks[i], y[i], weights[i]
    model = FoundationQueryHead().to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=3e-3)
    generator = torch.Generator().manual_seed(SEED + 100 + fold)
    loader = DataLoader(Rows(train_idx), batch_size=48, shuffle=True,
                        generator=generator, num_workers=2, pin_memory=True,
                        persistent_workers=True)
    best, best_auc, stale = None, -1.0, 0
    for epoch in range(24):
        model.train()
        for x, m, target, weight in loader:
            x, m = x.to(device), m.to(device)
            target, weight = target.to(device), weight.to(device)
            with torch.autocast("cuda", enabled=device.type == "cuda"):
                logits = model(x, m)
                raw = F.binary_cross_entropy_with_logits(logits, target,
                                                          reduction="none")
                loss = (raw * weight).sum() / weight.sum().clamp_min(1)
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        pred = predict_head(model, features, masks, val_idx, device)
        score = macro_auc(y[val_idx], pred)
        log(f"fold {fold} epoch {epoch}: grouped weak-val AUC {score:.5f}")
        if score > best_auc + 2e-4:
            best_auc, stale = score, 0
            best = {k: v.detach().cpu() for k, v in model.state_dict().items()}
        else:
            stale += 1
            if stale >= 5: break
    return best, best_auc


def _v52_rank_columns(values):
    frame = pd.DataFrame(np.asarray(values, dtype=np.float64))
    return frame.rank(method="average", pct=True).to_numpy(np.float64)


def _v52_validate_submission(frame, expected_ids):
    expected_columns = ["StudyInstanceUID", *TARGETS]
    if frame.columns.tolist() != expected_columns:
        raise RuntimeError("V52 submission schema drift")
    ids = frame["StudyInstanceUID"].astype(str).tolist()
    if ids != list(map(str, expected_ids)) or len(ids) != len(set(ids)):
        raise RuntimeError("V52 submission study identity/order drift")
    values = frame[TARGETS].to_numpy(np.float64)
    if not np.isfinite(values).all() or values.min() < 0 or values.max() > 1:
        raise RuntimeError("V52 submission values are invalid")


def main_v52():
    import shutil
    from sklearn.model_selection import GroupKFold

    output = Path("/kaggle/working/rsna_rad_e9")
    output.mkdir(parents=True, exist_ok=True)
    primary = Path("/kaggle/working/submission.csv")
    preserved = Path("/kaggle/working/submission_e2_preserved.csv")
    audit_path = Path("/kaggle/working/rad_e9_audit.json")
    audit = {
        "status": "E2_PRESERVED",
        "evidence_boundary": (
            "All OOF values are local diagnostics on 58 official image labels; "
            "they are not Kaggle competition scores. E2 remains the primary unless "
            "strict artifact, OOF, inference, and submission gates all pass."
        ),
        "encoder": "RadImageNet ResNet-50 official PyTorch release",
        "encoder_license": "CC-BY-NC-SA-4.0 (Kaggle-hosted weight metadata)",
        "encoder_sha256": "08629f7e7bd3e29b8ee9522ca3f65ce4d010a7ddf74f0ea3c7e3f3d0bbab0734",
        "encoder_source_commit": "0ce16f7375db4236e646829d1eca61cdb4282133",
        "base_oof_sha256": "62d47ba4c0c8347b5b24e7fd2aa517aae0fd6d4656fd5d829fd3a50f0159909c",
        "parent": "E2 captured 20-member DINOv2 rank ensemble",
        "blend_contract": "rank columns independently, then 80% E2 plus 20% RadImageNet",
        "pixel_rules": dict(RULES),
    }
    if not primary.is_file():
        raise FileNotFoundError("E2 parent submission is absent")
    shutil.copy2(primary, preserved)

    try:
        device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
        if device.type != "cuda":
            raise RuntimeError("V52 RadImageNet experiment requires CUDA")
        elapsed = max(0.0, time.time() - float(globals().get("T0", time.time())))
        available = 8.72 * 3600 - elapsed
        audit["elapsed_before_v52_seconds"] = elapsed
        audit["available_at_start_seconds"] = available
        if available < 2.0 * 3600:
            raise TimeoutError(f"only {available / 60:.1f} minutes remain")

        train = pd.read_csv(ROOT / "train.csv", dtype={"StudyInstanceUID": str})
        train_series = pd.read_csv(
            ROOT / "train_series.csv",
            dtype={"StudyInstanceUID": str, "SeriesInstanceUID": str},
        )
        if len(train) != 4407:
            raise RuntimeError(f"unexpected train study count {len(train)}")
        plane = dict(zip(train_series.SeriesInstanceUID, train_series.Anatomical_Plane))
        headers = annotate(walk("train_series"))
        audit_official_sequence_metadata(headers, train_series)
        studies, pixels, slot_mask = build_cache(
            pick_slots(headers, plane), plane, lat_of(headers, "train-v52 "), "train-v52"
        )
        by_uid = {str(uid): i for i, uid in enumerate(studies)}
        missing = [uid for uid in train.StudyInstanceUID if uid not in by_uid]
        if missing:
            raise RuntimeError(f"{len(missing)} train studies absent from cache")
        order = np.array([by_uid[uid] for uid in train.StudyInstanceUID], dtype=np.int64)
        pixels, slot_mask = pixels[order], slot_mask[order]
        train_token_count = int(np.repeat(slot_mask[:, :, None], CACHE_SLICES, 2).sum())
        if train_token_count < int(0.90 * len(train) * N_SLOT * CACHE_SLICES):
            raise RuntimeError(f"insufficient acquired train slices: {train_token_count}")
        features, token_mask = encode_radimagenet(pixels, slot_mask, device)
        del pixels, slot_mask, headers
        gc.collect()

        y, weights, gold = make_targets(train)
        if int(gold.sum()) != 58:
            raise RuntimeError(f"expected 58 fully gold studies, observed {int(gold.sum())}")
        groups = report_groups(train)
        if len(np.unique(groups)) < 4000:
            raise RuntimeError("unexpected report-group collapse")

        splits = list(GroupKFold(5).split(features, groups=groups))
        fold_id = np.full(len(train), -1, dtype=np.int8)
        folds = []
        oof = np.zeros_like(y, dtype=np.float32)
        for fold, (tr, va) in enumerate(splits):
            if set(groups[tr]).intersection(groups[va]):
                raise RuntimeError(f"report leakage in fold {fold}")
            fold_id[va] = fold
            state, score = train_fold(
                features, token_mask, y, weights, tr, va, fold, device
            )
            if state is None:
                raise RuntimeError(f"fold {fold} produced no checkpoint")
            head = FoundationQueryHead().to(device)
            head.load_state_dict(state, strict=True)
            oof[va] = predict_head(head, features, token_mask, va, device)
            folds.append({"fold": fold, "weak_auc": float(score), "state_dict": state})
            del head
            torch.cuda.empty_cache()
        if (fold_id < 0).any() or not np.isfinite(oof).all():
            raise RuntimeError("incomplete V52 OOF")

        weak_auc = macro_auc(y, oof)
        gold_auc = macro_auc(y[gold], oof[gold])
        log(f"V52 RadImageNet OOF weak macro AUC {weak_auc:.5f}")
        log(f"V52 RadImageNet OOF gold macro AUC {gold_auc:.5f} on 58 studies")
        torch.save(
            {
                "version": "v52-radimagenet-resnet50-official-1",
                "targets": TARGETS,
                "encoder_sha256": audit["encoder_sha256"],
                "encoder_source_commit": audit["encoder_source_commit"],
                "img": IMG,
                "slices_per_plane": CACHE_SLICES,
                "feature": "global_average_pool",
                "folds": folds,
                "weak_oof_auc": weak_auc,
                "gold_oof_auc": gold_auc,
            },
            output / "v52_radimagenet_heads.pt",
        )
        oof_frame = pd.DataFrame(oof, columns=TARGETS)
        oof_frame.insert(0, "StudyInstanceUID", train.StudyInstanceUID)
        oof_frame["fold"] = fold_id
        oof_frame["is_gold"] = gold.astype(np.uint8)
        oof_frame.to_csv(output / "v52_oof.csv", index=False)

        base_npz = find_input_file("oof.npz")
        observed_base_hash = _v52_sha256(base_npz)
        if observed_base_hash != audit["base_oof_sha256"]:
            raise RuntimeError(f"E2 OOF artifact drift: {observed_base_hash}")
        with np.load(base_npz, allow_pickle=False) as base_bundle:
            expected_members = {"ids", "pred", "y_derived", "gold_mask", "targets"}
            if set(base_bundle.files) != expected_members:
                raise RuntimeError(f"unexpected E2 OOF members: {base_bundle.files}")
            base_ids = base_bundle["ids"].astype(str)
            base_targets = base_bundle["targets"].astype(str).tolist()
            base_gold = base_bundle["gold_mask"].astype(bool)
            base_prediction = base_bundle["pred"].astype(np.float64)
        if base_targets != TARGETS:
            raise RuntimeError("E2 OOF target order drift")
        if not np.array_equal(base_ids, train.StudyInstanceUID.astype(str).to_numpy()):
            raise RuntimeError("E2 OOF study order drift")
        if not np.array_equal(base_gold, gold):
            raise RuntimeError("E2 OOF gold mask differs from official train.csv")
        train_rows = np.flatnonzero(gold)
        if len(train_rows) != 58:
            raise RuntimeError(f"expected 58 E2 gold rows, observed {len(train_rows)}")
        gold_y = train.loc[gold, TARGETS].to_numpy(np.float64)
        exact_public = base_prediction[gold]
        rad = oof[gold].astype(np.float64)
        if not all(np.isfinite(x).all() for x in (gold_y, exact_public, rad)):
            raise RuntimeError("non-finite aligned E2/RadImageNet OOF value")

        base_rank = _v52_rank_columns(exact_public)
        rad_rank = _v52_rank_columns(rad)
        base_score = macro_auc(gold_y, base_rank)
        rad_score = macro_auc(gold_y, rad_rank)
        alpha_grid = np.array([0.0, 0.025, 0.05, 0.10, 0.15, 0.20, 0.25])
        gold_folds = fold_id[train_rows]
        if sorted(np.unique(gold_folds).tolist()) != [0, 1, 2, 3, 4]:
            raise RuntimeError("gold rows do not cover all five grouped folds")
        nested = np.zeros_like(base_rank)
        choices = []
        outer_train_scores = []
        for outer in range(5):
            tr = gold_folds != outer
            va = ~tr
            scored = []
            for alpha in alpha_grid:
                blend = (1.0 - alpha) * base_rank[tr] + alpha * rad_rank[tr]
                score = macro_auc(gold_y[tr], blend) - 0.01 * float(alpha)
                scored.append(float(score))
            best = max(range(len(alpha_grid)), key=lambda i: (scored[i], -alpha_grid[i]))
            alpha = float(alpha_grid[best])
            choices.append(alpha)
            outer_train_scores.append(scored)
            nested[va] = (1.0 - alpha) * base_rank[va] + alpha * rad_rank[va]
        nested_score = macro_auc(gold_y, nested)
        # Deployment weight is fixed by the independently scored public 0.906 mechanism.
        # The 58 gold rows may veto it and measure fold stability, but do not tune it.
        final_alpha = 0.20
        final_oof = (1.0 - final_alpha) * base_rank + final_alpha * rad_rank
        final_score = macro_auc(gold_y, final_oof)
        grid_scores = {
            f"{alpha:.3f}": macro_auc(
                gold_y, (1.0 - alpha) * base_rank + alpha * rad_rank
            )
            for alpha in alpha_grid
        }
        base_target_scores = _v52_target_auc(gold_y, base_rank)
        rad_target_scores = _v52_target_auc(gold_y, rad_rank)
        final_target_scores = _v52_target_auc(gold_y, final_oof)
        target_deltas = {
            target: final_target_scores[target] - base_target_scores[target]
            for target in TARGETS
        }
        target_regressions = {
            target: delta for target, delta in target_deltas.items() if delta < -1e-12
        }
        positive_folds = int(sum(alpha > 0 for alpha in choices))
        supported = bool(
            final_alpha > 0
            and positive_folds >= 3
            and nested_score >= base_score + 0.001
            and final_score >= base_score + 0.001
            and not target_regressions
        )
        audit["oof"] = {
            "rows": 58,
            "weak_macro_auc": weak_auc,
            "rad_gold_macro_auc": rad_score,
            "e2_macro_auc": base_score,
            "outer_fold_choices": choices,
            "outer_fold_penalized_train_scores": outer_train_scores,
            "nested_blend_macro_auc": nested_score,
            "final_alpha": final_alpha,
            "final_descriptive_macro_auc": final_score,
            "full_grid_macro_auc": grid_scores,
            "positive_outer_folds": positive_folds,
            "per_target": {
                target: {
                    "e2_auc": base_target_scores[target],
                    "radimagenet_auc": rad_target_scores[target],
                    "blend_auc": final_target_scores[target],
                    "blend_delta": target_deltas[target],
                }
                for target in TARGETS
            },
            "target_regressions": target_regressions,
            "gold_fold_counts": {
                str(fold): int((gold_folds == fold).sum()) for fold in range(5)
            },
            "selection_supported": supported,
        }
        audit["train_available_slice_tokens"] = train_token_count
        audit["head_count"] = len(folds)
        if not supported:
            audit["status"] = "OOF_REJECTED_E2_PRESERVED"
            log(
                f"V52 rejected by nested OOF: base={base_score:.5f}, "
                f"nested={nested_score:.5f}, final={final_score:.5f}, choices={choices}"
            )
            return

        del features, token_mask
        gc.collect()
        test = pd.read_csv(ROOT / "test.csv", dtype={"StudyInstanceUID": str})
        test_series = pd.read_csv(
            ROOT / "test_series.csv",
            dtype={"StudyInstanceUID": str, "SeriesInstanceUID": str},
        )
        test_plane = dict(zip(test_series.SeriesInstanceUID, test_series.Anatomical_Plane))
        test_headers = annotate(walk("test_series"))
        audit_official_sequence_metadata(test_headers, test_series)
        test_studies, test_pixels, test_slot_mask = build_cache(
            pick_slots(test_headers, test_plane),
            test_plane,
            lat_of(test_headers, "test-v52 "),
            "test-v52",
        )
        test_by_uid = {str(uid): i for i, uid in enumerate(test_studies)}
        test_missing = [uid for uid in test.StudyInstanceUID if uid not in test_by_uid]
        if test_missing:
            raise RuntimeError(f"{len(test_missing)} test studies absent from cache")
        test_order = np.array([test_by_uid[uid] for uid in test.StudyInstanceUID])
        test_pixels = test_pixels[test_order]
        test_slot_mask = test_slot_mask[test_order]
        test_token_count = int(
            np.repeat(test_slot_mask[:, :, None], CACHE_SLICES, 2).sum()
        )
        if test_token_count < int(0.85 * len(test) * N_SLOT * CACHE_SLICES):
            raise RuntimeError(f"insufficient acquired test slices: {test_token_count}")
        test_features, test_token_mask = encode_radimagenet(
            test_pixels, test_slot_mask, device
        )
        del test_pixels, test_slot_mask, test_headers
        gc.collect()

        fold_predictions = []
        all_test = np.arange(len(test), dtype=np.int64)
        for record in folds:
            head = FoundationQueryHead().to(device)
            head.load_state_dict(record["state_dict"], strict=True)
            fold_predictions.append(
                predict_head(head, test_features, test_token_mask, all_test, device)
            )
            del head
            torch.cuda.empty_cache()
        if len(fold_predictions) != 5:
            raise RuntimeError("test inference did not use all five heads")
        rad_test = np.mean(np.stack(fold_predictions), axis=0)
        if not np.isfinite(rad_test).all():
            raise RuntimeError("non-finite RadImageNet test prediction")

        baseline = pd.read_csv(preserved, dtype={"StudyInstanceUID": str})
        _v52_validate_submission(baseline, test.StudyInstanceUID)
        rad_frame = pd.DataFrame(rad_test, columns=TARGETS)
        rad_frame.insert(0, "StudyInstanceUID", test.StudyInstanceUID)
        _v52_validate_submission(rad_frame, test.StudyInstanceUID)
        rad_frame.to_csv(output / "submission_rad_only.csv", index=False)
        baseline_rank = _v52_rank_columns(baseline[TARGETS].to_numpy())
        rad_test_rank = _v52_rank_columns(rad_test)
        selected_path = None
        for alpha in alpha_grid[1:]:
            candidate = baseline.copy()
            candidate[TARGETS] = (
                (1.0 - alpha) * baseline_rank + alpha * rad_test_rank
            )
            _v52_validate_submission(candidate, test.StudyInstanceUID)
            path = output / f"submission_e2_rad_{int(round(1000 * alpha)):03d}.csv"
            candidate.to_csv(path, index=False)
            if abs(float(alpha) - final_alpha) < 1e-12:
                selected_path = path
        if selected_path is None or not selected_path.is_file():
            raise RuntimeError("selected V52 blend artifact is absent")
        selected = pd.read_csv(selected_path, dtype={"StudyInstanceUID": str})
        _v52_validate_submission(selected, test.StudyInstanceUID)
        audit["test_studies"] = len(test)
        audit["test_available_slice_tokens"] = test_token_count
        audit["test_head_count"] = len(fold_predictions)
        audit["selected_path"] = str(selected_path)
        audit["selected_sha256"] = _v52_sha256(selected_path)
        audit["fallback_sha256"] = _v52_sha256(preserved)
        shutil.copy2(selected_path, primary)
        if _v52_sha256(primary) != audit["selected_sha256"]:
            raise RuntimeError("primary V52 copy hash mismatch")
        audit["status"] = "CANDIDATE_SELECTED"
        log(
            f"E9 selected alpha={final_alpha:.3f}; "
            f"nested={nested_score:.5f} vs E2 OOF={base_score:.5f}"
        )
    except Exception as error:
        audit["status"] = "ERROR_E2_PRESERVED"
        audit["error"] = f"{type(error).__name__}: {error}"
        audit["traceback"] = traceback.format_exc()
        log(f"E9 preserves E2: {audit['error']}")
    finally:
        if audit.get("status") != "CANDIDATE_SELECTED" and preserved.is_file():
            shutil.copy2(preserved, primary)
        audit["primary_sha256"] = _v52_sha256(primary) if primary.is_file() else None
        audit_path.write_text(json.dumps(audit, indent=2, sort_keys=True) + "\n")


def _v52_load_pinned_e9b():
    """Load the public v15 heads and reconstruct the dual-OOF target gate."""
    heads_path = find_input_file("v52_radimagenet_heads.pt")
    remote_path = find_input_file("rad_e9_audit.json")
    public_path = find_input_file("public_oof_diagnostic.json")
    contract_path = find_input_file("e9b_contract.json")
    expected_hashes = {
        heads_path: PINNED_HEADS_SHA256,
        remote_path: PINNED_REMOTE_AUDIT_SHA256,
        public_path: PINNED_PUBLIC_DIAGNOSTIC_SHA256,
        contract_path: PINNED_E9B_CONTRACT_SHA256,
    }
    for path, expected in expected_hashes.items():
        observed = _v52_sha256(path)
        if observed != expected:
            raise RuntimeError(f"pinned E9b artifact drift for {path.name}: {observed}")

    remote = json.loads(remote_path.read_text())
    public = json.loads(public_path.read_text())
    contract = json.loads(contract_path.read_text())
    if remote.get("status") != "OOF_REJECTED_E2_PRESERVED":
        raise RuntimeError(f"unexpected v15 audit status {remote.get('status')}")
    if remote.get("primary_sha256") != (
        "f9fb57b7bac8489a5d5285b3984b06df57f142572be6417eac6341c43e96707a"
    ):
        raise RuntimeError("v15 did not preserve the exact E2 visible artifact")
    remote_oof = remote.get("oof", {})
    public_target = public.get("per_target", {})
    remote_target = remote_oof.get("per_target", {})
    if set(public_target) != set(TARGETS) or set(remote_target) != set(TARGETS):
        raise RuntimeError("E9b diagnostic target set drift")
    if int(remote.get("head_count", -1)) != 5:
        raise RuntimeError("v15 remote audit does not contain five heads")
    if int(remote_oof.get("positive_outer_folds", -1)) != 5:
        raise RuntimeError("v15 remote outer-fold support drift")
    if int(public.get("positive_outer_folds", -1)) != 5:
        raise RuntimeError("public outer-fold support drift")

    selected_targets = [
        target for target in TARGETS
        if float(public_target[target]["blend_delta"]) > 1e-12
        and float(remote_target[target]["blend_delta"]) > 1e-12
    ]
    preserved_targets = [target for target in TARGETS if target not in selected_targets]
    if selected_targets != contract.get("selected_targets"):
        raise RuntimeError(f"E9b selected-target contract drift: {selected_targets}")
    if preserved_targets != contract.get("preserved_targets"):
        raise RuntimeError(f"E9b preserved-target contract drift: {preserved_targets}")
    if len(selected_targets) != 10 or preserved_targets != ["Baker's", "Fracture"]:
        raise RuntimeError("E9b requires the ten-target dual-OOF intersection")
    alpha = float(contract.get("alpha", -1))
    if abs(alpha - 0.20) > 1e-12:
        raise RuntimeError(f"E9b alpha drift: {alpha}")

    def selective_macro(records, base_key, blend_key):
        return float(np.mean([
            float(records[target][blend_key] if target in selected_targets
                  else records[target][base_key])
            for target in TARGETS
        ]))

    public_base = float(public["base_gold_macro_auc"])
    remote_base = float(remote_oof["e2_macro_auc"])
    public_selective = selective_macro(public_target, "base_auc", "blend_auc")
    remote_selective = selective_macro(remote_target, "e2_auc", "blend_auc")
    if public_selective < public_base + 0.001:
        raise RuntimeError("E9b public selective gate no longer improves E2")
    if remote_selective < remote_base + 0.001:
        raise RuntimeError("E9b remote selective gate no longer improves E2")

    payload = torch.load(heads_path, map_location="cpu", weights_only=True)
    expected_payload = {
        "version": "v52-radimagenet-resnet50-official-1",
        "targets": TARGETS,
        "encoder_sha256": (
            "08629f7e7bd3e29b8ee9522ca3f65ce4d010a7ddf74f0ea3c7e3f3d0bbab0734"
        ),
        "encoder_source_commit": "0ce16f7375db4236e646829d1eca61cdb4282133",
        "img": 224,
        "slices_per_plane": 8,
        "feature": "global_average_pool",
    }
    for key, expected in expected_payload.items():
        if payload.get(key) != expected:
            raise RuntimeError(f"pinned E9b head contract drift for {key}")
    folds = payload.get("folds")
    if not isinstance(folds, list) or len(folds) != 5:
        raise RuntimeError("pinned E9b payload requires five folds")
    if sorted(int(record.get("fold", -1)) for record in folds) != list(range(5)):
        raise RuntimeError("pinned E9b fold identity drift")
    if any(not isinstance(record.get("state_dict"), dict) for record in folds):
        raise RuntimeError("pinned E9b state dictionary is absent")
    if abs(float(payload.get("weak_oof_auc", -1)) - 0.8278261335825697) > 1e-12:
        raise RuntimeError("pinned E9b weak OOF drift")
    if abs(float(payload.get("gold_oof_auc", -1)) - 0.8543239133509962) > 1e-12:
        raise RuntimeError("pinned E9b gold OOF drift")
    return {
        "payload": payload,
        "folds": folds,
        "alpha": alpha,
        "selected_targets": selected_targets,
        "preserved_targets": preserved_targets,
        "public_base": public_base,
        "public_selective": public_selective,
        "remote_base": remote_base,
        "remote_selective": remote_selective,
        "remote_outer_fold_choices": remote_oof["outer_fold_choices"],
    }


def _v52_e10_remote_ladder(contract):
    """Recompute this account's half of the ladder from artifacts the kernel can read.

    The contract carries per-target gains for two independent RadImageNet OOF runs. Only the
    public run is unverifiable here, so its numbers stay data. The remote run is rebuilt from
    the attached OOF table, the pinned E2 OOF bundle and the official labels, and must match
    the contract exactly or E10 refuses to deploy.
    """
    train = pd.read_csv(ROOT / "train.csv", dtype={"StudyInstanceUID": str})
    gold = train[TARGETS].notna().all(axis=1).to_numpy()
    oof_path = find_input_file("v52_oof.csv")
    observed = _v52_sha256(oof_path)
    if observed != contract["remote_oof_sha256"]:
        raise RuntimeError(f"E10 remote OOF drift: {observed}")
    rad_frame = pd.read_csv(oof_path, dtype={"StudyInstanceUID": str})
    if rad_frame.columns.tolist() != ["StudyInstanceUID", *TARGETS, "fold", "is_gold"]:
        raise RuntimeError("E10 remote OOF schema drift")
    aligned = train[["StudyInstanceUID"]].merge(
        rad_frame, on="StudyInstanceUID", how="left", validate="one_to_one"
    )
    if aligned[TARGETS].isna().any().any():
        raise RuntimeError("E10 remote OOF does not cover every official train study")

    base_npz = find_input_file("oof.npz")
    with np.load(base_npz, allow_pickle=False) as bundle:
        if bundle["targets"].astype(str).tolist() != TARGETS:
            raise RuntimeError("E10 E2 OOF target order drift")
        if not np.array_equal(
            bundle["ids"].astype(str), train.StudyInstanceUID.astype(str).to_numpy()
        ):
            raise RuntimeError("E10 E2 OOF study order drift")
        if not np.array_equal(bundle["gold_mask"].astype(bool), gold):
            raise RuntimeError("E10 E2 gold mask differs from official train.csv")
        base_prediction = bundle["pred"].astype(np.float64)

    # Rank within the scored rows, matching both the E9b parent and test-time deployment
    # where the ranked population and the scored population are the same studies.
    base = _v52_rank_columns(base_prediction[gold])
    rad = _v52_rank_columns(aligned[TARGETS].to_numpy(np.float64)[gold])
    gold_y = train.loc[gold, TARGETS].to_numpy(np.float64)
    if len(gold_y) != 58 or not np.isfinite(base).all() or not np.isfinite(rad).all():
        raise RuntimeError("E10 gold alignment is incomplete or non-finite")
    reference = _v52_target_auc(gold_y, base)
    if abs(
        float(np.mean([reference[t] for t in TARGETS]))
        - float(contract["base_gold_macro_auc"])
    ) > 1e-9:
        raise RuntimeError("E10 base gold diagnostic drift")

    rebuilt = {}
    for key in contract["ladder"]:
        alpha = float(key)
        scores = _v52_target_auc(gold_y, (1.0 - alpha) * base + alpha * rad)
        rebuilt[key] = {t: scores[t] - reference[t] for t in TARGETS}
    pinned_remote = contract["per_target_ladder_delta"]["remote_v15"]
    if set(rebuilt) != set(pinned_remote):
        raise RuntimeError("E10 ladder key drift")
    for key, deltas in rebuilt.items():
        for target, delta in deltas.items():
            if abs(delta - float(pinned_remote[key][target])) > 1e-9:
                raise RuntimeError(
                    f"E10 recomputed remote gain disagrees at {key}/{target}: {delta}"
                )
    if any(abs(reference[t] - float(contract["per_target_base_auc"][t])) > 1e-9 for t in TARGETS):
        raise RuntimeError("E10 per-target base AUC drift")
    return rebuilt, reference


def _v52_load_e10():
    """Validate the E10 contract, then return the weight map the kernel will deploy."""
    heads_path = find_input_file("v52_radimagenet_heads.pt")
    remote_path = find_input_file("rad_e9_audit.json")
    contract_path = find_input_file("e10_contract.json")
    for path, expected in (
        (heads_path, PINNED_HEADS_SHA256),
        (remote_path, PINNED_REMOTE_AUDIT_SHA256),
        (contract_path, PINNED_E10_CONTRACT_SHA256),
    ):
        observed = _v52_sha256(path)
        if observed != expected:
            raise RuntimeError(f"pinned E10 artifact drift for {path.name}: {observed}")

    contract = json.loads(contract_path.read_text())
    if contract.get("version") != "e10-alpha-ladder-2":
        raise RuntimeError(f"unexpected E10 contract version {contract.get('version')}")
    if contract.get("targets") != TARGETS:
        raise RuntimeError("E10 contract target order drift")
    remote = json.loads(remote_path.read_text())
    if remote.get("status") != "OOF_REJECTED_E2_PRESERVED":
        raise RuntimeError(f"unexpected v15 audit status {remote.get('status')}")
    if remote.get("primary_sha256") != (
        "f9fb57b7bac8489a5d5285b3984b06df57f142572be6417eac6341c43e96707a"
    ):
        raise RuntimeError("v15 did not preserve the exact E2 visible artifact")
    if int(remote.get("head_count", -1)) != 5:
        raise RuntimeError("v15 remote audit does not contain five heads")

    rebuilt, base_auc = _v52_e10_remote_ladder(contract)
    public_ladder = contract["per_target_ladder_delta"]["public"]
    configuration = contract["configurations"].get(E10_CONFIG)
    if configuration is None:
        raise RuntimeError(f"E10 contract has no configuration {E10_CONFIG!r}")
    alpha_map = {t: float(configuration["alpha_map"][t]) for t in TARGETS}
    if any(alpha < 0.0 or alpha > 1.0 for alpha in alpha_map.values()):
        raise RuntimeError("E10 weight outside the unit interval")
    preserved = sorted(t for t, alpha in alpha_map.items() if alpha == 0.0)
    if preserved != sorted(E10_PRESERVED_TARGETS):
        raise RuntimeError(f"E10 preserved-target drift: {preserved}")

    # Two-tier gate. The scored objective is macro AUC, so the binding requirement is that
    # the deployed map raise the macro in BOTH independent runs -- the public numbers as
    # pinned data, this account's numbers as recomputed above. A per-target "never harm any
    # single label" rule is strictly stronger than that objective and would veto rungs that
    # trade a small loss on one label for a large gain on another, so it is enforced only for
    # configurations that actually claim it. Whichever claim the contract makes is verified;
    # a configuration cannot quietly assert dual-positivity it no longer has.
    claims_dual_positive = bool(configuration["all_dual_positive"])
    observed_dual_positive = True
    for target, alpha in alpha_map.items():
        if alpha == 0.0:
            continue
        key = f"{alpha:.2f}"
        if key not in rebuilt:
            raise RuntimeError(f"E10 weight {key} is outside the audited ladder")
        gains = (float(public_ladder[key][target]), float(rebuilt[key][target]))
        if not all(gain > 0 for gain in gains):
            observed_dual_positive = False
            if claims_dual_positive:
                raise RuntimeError(
                    f"E10 dual-source gate rejects {target} at {key}: {gains}"
                )
    if observed_dual_positive != claims_dual_positive:
        raise RuntimeError(
            f"E10 contract claims all_dual_positive={claims_dual_positive} for "
            f"{E10_CONFIG!r} but recomputation observes {observed_dual_positive}"
        )

    macro = {}
    for name, ladder in (("public", public_ladder), ("remote_v15", rebuilt)):
        total = 0.0
        for target, alpha in alpha_map.items():
            gain = 0.0 if alpha == 0.0 else float(ladder[f"{alpha:.2f}"][target])
            total += float(base_auc[target]) + gain
        macro[name] = total / len(TARGETS)
        if macro[name] <= float(contract["base_gold_macro_auc"]):
            raise RuntimeError(
                f"E10 macro gate rejects {E10_CONFIG!r}: {name} macro {macro[name]} "
                f"does not beat base {contract['base_gold_macro_auc']}"
            )
        if abs(macro[name] - float(configuration["descriptive_macro"][name])) > 1e-9:
            raise RuntimeError(
                f"E10 recomputed {name} macro {macro[name]} disagrees with the contract "
                f"value {configuration['descriptive_macro'][name]}"
            )

    payload = torch.load(heads_path, map_location="cpu", weights_only=True)
    expected_payload = {
        "version": "v52-radimagenet-resnet50-official-1",
        "targets": TARGETS,
        "encoder_sha256": (
            "08629f7e7bd3e29b8ee9522ca3f65ce4d010a7ddf74f0ea3c7e3f3d0bbab0734"
        ),
        "encoder_source_commit": "0ce16f7375db4236e646829d1eca61cdb4282133",
        "img": 224,
        "slices_per_plane": 8,
        "feature": "global_average_pool",
    }
    for key, expected in expected_payload.items():
        if payload.get(key) != expected:
            raise RuntimeError(f"pinned E10 head contract drift for {key}")
    folds = payload.get("folds")
    if not isinstance(folds, list) or len(folds) != 5:
        raise RuntimeError("pinned E10 payload requires five folds")
    if sorted(int(record.get("fold", -1)) for record in folds) != list(range(5)):
        raise RuntimeError("pinned E10 fold identity drift")
    if any(not isinstance(record.get("state_dict"), dict) for record in folds):
        raise RuntimeError("pinned E10 state dictionary is absent")
    if abs(float(payload.get("gold_oof_auc", -1)) - 0.8543239133509962) > 1e-12:
        raise RuntimeError("pinned E10 gold OOF drift")
    return {
        "payload": payload,
        "folds": folds,
        "contract": contract,
        "configuration": E10_CONFIG,
        "alpha_map": alpha_map,
        "preserved_targets": sorted(E10_PRESERVED_TARGETS),
        "diagnostic_macro": configuration["diagnostic_macro"],
        "recomputed_macro": macro,
        "all_dual_positive": claims_dual_positive,
        "rationale": configuration["rationale"],
        "recomputed_remote_ladder": rebuilt,
    }


def main_v52_pinned_e9b():
    """Inference-only E9b from hash-pinned v15 heads; preserve E2 on any failure."""
    import shutil

    output = Path("/kaggle/working/rsna_rad_e9b")
    output.mkdir(parents=True, exist_ok=True)
    primary = Path("/kaggle/working/submission.csv")
    preserved = Path("/kaggle/working/submission_e2_preserved.csv")
    audit_path = Path("/kaggle/working/rad_e9b_audit.json")
    audit = {
        "status": "E2_PRESERVED",
        "mode": "pinned_v15_heads_inference_only",
        "evidence_boundary": (
            "OOF values are diagnostics, not Kaggle scores. The fixed public 20-percent "
            "vote is applied only to targets improving in two independent OOF runs."
        ),
        "encoder": "RadImageNet ResNet-50 official PyTorch release",
        "encoder_license": "CC-BY-NC-SA-4.0",
        "encoder_sha256": (
            "08629f7e7bd3e29b8ee9522ca3f65ce4d010a7ddf74f0ea3c7e3f3d0bbab0734"
        ),
        "heads_sha256": PINNED_HEADS_SHA256,
        "parent": "E2 captured 20-member DINOv2 rank ensemble",
        "pixel_rules": dict(RULES),
    }
    if not primary.is_file():
        raise FileNotFoundError("E2 parent submission is absent")
    shutil.copy2(primary, preserved)

    try:
        device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
        if device.type != "cuda":
            raise RuntimeError("E9b RadImageNet inference requires CUDA")
        elapsed = max(0.0, time.time() - float(globals().get("T0", time.time())))
        available = 8.72 * 3600 - elapsed
        audit["elapsed_before_e9b_seconds"] = elapsed
        audit["available_at_start_seconds"] = available
        if available < 45 * 60:
            raise TimeoutError(f"only {available / 60:.1f} minutes remain")

        pinned = _v52_load_pinned_e9b()
        audit["oof_gate"] = {
            "alpha": pinned["alpha"],
            "selected_targets": pinned["selected_targets"],
            "preserved_targets": pinned["preserved_targets"],
            "public_e2_macro_auc": pinned["public_base"],
            "public_selective_macro_auc": pinned["public_selective"],
            "remote_e2_macro_auc": pinned["remote_base"],
            "remote_selective_macro_auc": pinned["remote_selective"],
            "remote_outer_fold_choices": pinned["remote_outer_fold_choices"],
            "selection_supported": True,
        }

        test = pd.read_csv(ROOT / "test.csv", dtype={"StudyInstanceUID": str})
        test_series = pd.read_csv(
            ROOT / "test_series.csv",
            dtype={"StudyInstanceUID": str, "SeriesInstanceUID": str},
        )
        plane = dict(zip(test_series.SeriesInstanceUID, test_series.Anatomical_Plane))
        headers = annotate(walk("test_series"))
        audit_official_sequence_metadata(headers, test_series)
        studies, pixels, slot_mask = build_cache(
            pick_slots(headers, plane), plane, lat_of(headers, "test-e9b "), "test-e9b"
        )
        by_uid = {str(uid): index for index, uid in enumerate(studies)}
        missing = [uid for uid in test.StudyInstanceUID if uid not in by_uid]
        if missing:
            raise RuntimeError(f"{len(missing)} test studies absent from E9b cache")
        order = np.asarray([by_uid[uid] for uid in test.StudyInstanceUID], dtype=np.int64)
        pixels, slot_mask = pixels[order], slot_mask[order]
        token_count = int(np.repeat(slot_mask[:, :, None], CACHE_SLICES, 2).sum())
        if token_count < int(0.85 * len(test) * N_SLOT * CACHE_SLICES):
            raise RuntimeError(f"insufficient acquired E9b test slices: {token_count}")
        features, token_mask = encode_radimagenet(pixels, slot_mask, device)
        del pixels, slot_mask, headers
        gc.collect()

        all_test = np.arange(len(test), dtype=np.int64)
        fold_predictions = []
        for record in pinned["folds"]:
            head = FoundationQueryHead().to(device)
            head.load_state_dict(record["state_dict"], strict=True)
            fold_predictions.append(
                predict_head(head, features, token_mask, all_test, device)
            )
            del head
            torch.cuda.empty_cache()
        if len(fold_predictions) != 5:
            raise RuntimeError("E9b test inference did not use all five heads")
        rad_test = np.mean(np.stack(fold_predictions), axis=0)
        if rad_test.shape != (len(test), len(TARGETS)) or not np.isfinite(rad_test).all():
            raise RuntimeError(f"invalid E9b prediction shape/value: {rad_test.shape}")

        baseline = pd.read_csv(preserved, dtype={"StudyInstanceUID": str})
        _v52_validate_submission(baseline, test.StudyInstanceUID)
        rad_frame = pd.DataFrame(rad_test, columns=TARGETS)
        rad_frame.insert(0, "StudyInstanceUID", test.StudyInstanceUID)
        _v52_validate_submission(rad_frame, test.StudyInstanceUID)
        rad_frame.to_csv(output / "submission_rad_only.csv", index=False)
        baseline_rank = _v52_rank_columns(baseline[TARGETS].to_numpy())
        rad_rank = _v52_rank_columns(rad_test)
        selected = baseline.copy()
        alpha = pinned["alpha"]
        for target in pinned["selected_targets"]:
            index = TARGETS.index(target)
            selected[target] = (
                (1.0 - alpha) * baseline_rank[:, index] + alpha * rad_rank[:, index]
            )
        for target in pinned["preserved_targets"]:
            if not np.array_equal(
                selected[target].to_numpy(), baseline[target].to_numpy()
            ):
                raise RuntimeError(f"E9b failed to preserve {target}")
        _v52_validate_submission(selected, test.StudyInstanceUID)
        selected_path = output / "submission_e2_rad_robust_200.csv"
        selected.to_csv(selected_path, index=False)
        selected = pd.read_csv(selected_path, dtype={"StudyInstanceUID": str})
        _v52_validate_submission(selected, test.StudyInstanceUID)

        audit.update({
            "test_studies": len(test),
            "test_available_slice_tokens": token_count,
            "test_head_count": len(fold_predictions),
            "selected_path": str(selected_path),
            "selected_sha256": _v52_sha256(selected_path),
            "fallback_sha256": _v52_sha256(preserved),
        })
        shutil.copy2(selected_path, primary)
        if _v52_sha256(primary) != audit["selected_sha256"]:
            raise RuntimeError("primary E9b copy hash mismatch")
        audit["status"] = "CANDIDATE_SELECTED"
        log(
            f"E9b selected alpha={alpha:.3f} on "
            f"{len(pinned['selected_targets'])} dual-OOF-stable targets; "
            "Baker's and Fracture preserve E2"
        )
    except Exception as error:
        audit["status"] = "ERROR_E2_PRESERVED"
        audit["error"] = f"{type(error).__name__}: {error}"
        audit["traceback"] = traceback.format_exc()
        log(f"E9b preserves E2: {audit['error']}")
    finally:
        if audit.get("status") != "CANDIDATE_SELECTED" and preserved.is_file():
            shutil.copy2(preserved, primary)
        audit["primary_sha256"] = _v52_sha256(primary) if primary.is_file() else None
        audit_path.write_text(json.dumps(audit, indent=2, sort_keys=True) + "\n")


def _v52_rad_test_predictions(pinned, test, device, tag):
    """Five-head RadImageNet test prediction on the official test tree."""
    test_series = pd.read_csv(
        ROOT / "test_series.csv",
        dtype={"StudyInstanceUID": str, "SeriesInstanceUID": str},
    )
    plane = dict(zip(test_series.SeriesInstanceUID, test_series.Anatomical_Plane))
    headers = annotate(walk("test_series"))
    audit_official_sequence_metadata(headers, test_series)
    studies, pixels, slot_mask = build_cache(
        pick_slots(headers, plane), plane, lat_of(headers, f"{tag} "), tag
    )
    by_uid = {str(uid): index for index, uid in enumerate(studies)}
    missing = [uid for uid in test.StudyInstanceUID if uid not in by_uid]
    if missing:
        raise RuntimeError(f"{len(missing)} test studies absent from {tag} cache")
    order = np.asarray([by_uid[uid] for uid in test.StudyInstanceUID], dtype=np.int64)
    pixels, slot_mask = pixels[order], slot_mask[order]
    token_count = int(np.repeat(slot_mask[:, :, None], CACHE_SLICES, 2).sum())
    if token_count < int(0.85 * len(test) * N_SLOT * CACHE_SLICES):
        raise RuntimeError(f"insufficient acquired {tag} test slices: {token_count}")
    features, token_mask = encode_radimagenet(pixels, slot_mask, device)
    del pixels, slot_mask, headers
    gc.collect()

    rows = np.arange(len(test), dtype=np.int64)
    predictions = []
    for record in pinned["folds"]:
        head = FoundationQueryHead().to(device)
        head.load_state_dict(record["state_dict"], strict=True)
        predictions.append(predict_head(head, features, token_mask, rows, device))
        del head
        torch.cuda.empty_cache()
    if len(predictions) != 5:
        raise RuntimeError(f"{tag} test inference did not use all five heads")
    rad_test = np.mean(np.stack(predictions), axis=0)
    if rad_test.shape != (len(test), len(TARGETS)) or not np.isfinite(rad_test).all():
        raise RuntimeError(f"invalid {tag} prediction shape/value: {rad_test.shape}")
    return rad_test, token_count, len(predictions)


def main_v52_e10():
    """Deploy the audited E10 weight map; preserve the E2 parent on any failure."""
    import shutil

    output = Path("/kaggle/working/rsna_rad_e10")
    output.mkdir(parents=True, exist_ok=True)
    primary = Path("/kaggle/working/submission.csv")
    preserved = Path("/kaggle/working/submission_e2_preserved.csv")
    audit_path = Path("/kaggle/working/rad_e10_audit.json")
    audit = {
        "status": "E2_PRESERVED",
        "mode": "pinned_v15_heads_inference_only",
        "experiment": "E10",
        "configuration": E10_CONFIG,
        "evidence_boundary": (
            "OOF values are 58-study diagnostics on official train labels, not Kaggle "
            "scores. E10 widens the weight ladder that E9 truncated at 0.25 and votes only "
            "where two independent OOF runs agree at the deployed weight."
        ),
        "encoder": "RadImageNet ResNet-50 official PyTorch release",
        "encoder_license": "CC-BY-NC-SA-4.0",
        "heads_sha256": PINNED_HEADS_SHA256,
        "contract_sha256": PINNED_E10_CONTRACT_SHA256,
        "parent": "E2 captured 20-member DINOv2 rank ensemble",
        "pixel_rules": dict(RULES),
    }
    if not primary.is_file():
        raise FileNotFoundError("E2 parent submission is absent")
    shutil.copy2(primary, preserved)

    try:
        device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
        if device.type != "cuda":
            raise RuntimeError("E10 RadImageNet inference requires CUDA")
        elapsed = max(0.0, time.time() - float(globals().get("T0", time.time())))
        available = 8.72 * 3600 - elapsed
        audit["elapsed_before_e10_seconds"] = elapsed
        audit["available_at_start_seconds"] = available
        if available < 45 * 60:
            raise TimeoutError(f"only {available / 60:.1f} minutes remain")

        pinned = _v52_load_e10()
        audit["weight_gate"] = {
            "configuration": pinned["configuration"],
            "alpha_map": pinned["alpha_map"],
            "preserved_targets": pinned["preserved_targets"],
            "diagnostic_macro": pinned["diagnostic_macro"],
            "recomputed_macro": pinned["recomputed_macro"],
            "all_dual_positive": pinned["all_dual_positive"],
            "base_gold_macro_auc": pinned["contract"]["base_gold_macro_auc"],
            "rationale": pinned["rationale"],
            "remote_ladder_recomputed_in_kernel": True,
        }

        test = pd.read_csv(ROOT / "test.csv", dtype={"StudyInstanceUID": str})
        rad_test, token_count, head_count = _v52_rad_test_predictions(
            pinned, test, device, "test-e10"
        )

        baseline = pd.read_csv(preserved, dtype={"StudyInstanceUID": str})
        _v52_validate_submission(baseline, test.StudyInstanceUID)
        rad_frame = pd.DataFrame(rad_test, columns=TARGETS)
        rad_frame.insert(0, "StudyInstanceUID", test.StudyInstanceUID)
        _v52_validate_submission(rad_frame, test.StudyInstanceUID)
        rad_frame.to_csv(output / "submission_rad_only.csv", index=False)
        baseline_rank = _v52_rank_columns(baseline[TARGETS].to_numpy())
        rad_rank = _v52_rank_columns(rad_test)

        # Materialise every audited rung so the ladder is inspectable from one run; only the
        # configured map is promoted to the visible submission.
        written = {}
        for name, configuration in sorted(pinned["contract"]["configurations"].items()):
            frame = baseline.copy()
            for target, alpha in configuration["alpha_map"].items():
                alpha = float(alpha)
                if alpha > 0:
                    index = TARGETS.index(target)
                    frame[target] = (
                        (1.0 - alpha) * baseline_rank[:, index] + alpha * rad_rank[:, index]
                    )
            for target, alpha in configuration["alpha_map"].items():
                if float(alpha) == 0.0 and not np.array_equal(
                    frame[target].to_numpy(), baseline[target].to_numpy()
                ):
                    raise RuntimeError(f"E10 failed to preserve {target} in {name}")
            _v52_validate_submission(frame, test.StudyInstanceUID)
            path = output / f"submission_e10_{name}.csv"
            frame.to_csv(path, index=False)
            written[name] = _v52_sha256(path)
        audit["ladder_sha256"] = written

        selected_path = output / f"submission_e10_{pinned['configuration']}.csv"
        selected = pd.read_csv(selected_path, dtype={"StudyInstanceUID": str})
        _v52_validate_submission(selected, test.StudyInstanceUID)
        for target in pinned["preserved_targets"]:
            if not np.array_equal(
                selected[target].to_numpy(), baseline[target].to_numpy()
            ):
                raise RuntimeError(f"E10 promoted file does not preserve {target}")
        audit.update({
            "test_studies": len(test),
            "test_available_slice_tokens": token_count,
            "test_head_count": head_count,
            "selected_path": str(selected_path),
            "selected_sha256": _v52_sha256(selected_path),
            "fallback_sha256": _v52_sha256(preserved),
        })
        shutil.copy2(selected_path, primary)
        if _v52_sha256(primary) != audit["selected_sha256"]:
            raise RuntimeError("primary E10 copy hash mismatch")
        audit["status"] = "CANDIDATE_SELECTED"
        voted = sorted(t for t, alpha in pinned["alpha_map"].items() if alpha > 0)
        log(
            f"E10 promoted {pinned['configuration']} over {len(voted)} dual-OOF targets; "
            f"{', '.join(pinned['preserved_targets'])} preserve E2"
        )
    except Exception as error:
        audit["status"] = "ERROR_E2_PRESERVED"
        audit["error"] = f"{type(error).__name__}: {error}"
        audit["traceback"] = traceback.format_exc()
        log(f"E10 preserves E2: {audit['error']}")
    finally:
        if audit.get("status") != "CANDIDATE_SELECTED" and preserved.is_file():
            shutil.copy2(preserved, primary)
        audit["primary_sha256"] = _v52_sha256(primary) if primary.is_file() else None
        audit_path.write_text(json.dumps(audit, indent=2, sort_keys=True) + "\n")


def _v52_e11_availability(headers, plane_map):
    """Count studies offering each (plane, fat-suppression) pair before any slot is picked.

    The parent arm reads only fat-suppressed series and never had to ask how many studies
    carry a non-suppressed one. E11 depends on that answer, so it is measured and logged
    rather than assumed: a slot nobody can fill is a masked column, and a run that produced
    one silently would look like a weak arm instead of an absent input.
    """
    frame = headers[["StudyInstanceUID", "SeriesInstanceUID", "fatsat"]].copy()
    frame["plane"] = frame.SeriesInstanceUID.map(plane_map)
    total = frame.StudyInstanceUID.nunique()
    table = {}
    for plane in ("Sagittal", "Coronal", "Axial"):
        for fatsat in (True, False):
            selected = frame[(frame.plane == plane) & (frame.fatsat == bool(fatsat))]
            studies = selected.StudyInstanceUID.nunique()
            key = f"{plane}_{'FS' if fatsat else 'NOFS'}"
            table[key] = {
                "studies": int(studies),
                "fraction": float(studies / total) if total else 0.0,
                "series": int(len(selected)),
            }
            log(f"E11 availability {key}: {studies}/{total} studies, {len(selected)} series")
    return table


def main_v52_e11():
    """Train a third arm on a deliberately different pixel recipe. Never ships a candidate."""
    import shutil
    from sklearn.model_selection import GroupKFold

    output = Path("/kaggle/working/rsna_rad_e11")
    output.mkdir(parents=True, exist_ok=True)
    primary = Path("/kaggle/working/submission.csv")
    preserved = Path("/kaggle/working/submission_e2_preserved.csv")
    audit_path = Path("/kaggle/working/rad_e11_audit.json")
    audit = {
        "status": "E2_PRESERVED",
        "mode": "e11-diverse-recipe-training-only",
        "evidence_boundary": (
            "Every value here is a local out-of-fold diagnostic on 58 official image "
            "labels. It is not a Kaggle score, and this mode never replaces the parent "
            "submission under any outcome."
        ),
        "recipe": {
            "slots": [list(slot) for slot in E11_SLOTS],
            "crop_mm": E11_CROP_MM,
            "cache_slices": E11_CACHE_SLICES,
            "img": E11_IMG,
            "differs_from_parent_arm": (
                "parent reads 3 fat-suppressed slots at full frame; this reads 3 "
                "non-suppressed slots plus 1 suppressed anchor at a 130 mm physical crop"
            ),
        },
        "encoder": "RadImageNet ResNet-50 official PyTorch release",
        "encoder_license": "CC-BY-NC-SA-4.0 (Kaggle-hosted weight metadata)",
        "encoder_sha256": "08629f7e7bd3e29b8ee9522ca3f65ce4d010a7ddf74f0ea3c7e3f3d0bbab0734",
    }
    if not primary.is_file():
        raise FileNotFoundError("E2 parent submission is absent")
    shutil.copy2(primary, preserved)

    try:
        device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
        if device.type != "cuda":
            raise RuntimeError("E11 training requires CUDA")
        elapsed = max(0.0, time.time() - float(globals().get("T0", time.time())))
        available = TIME_BUDGET - elapsed
        audit["elapsed_before_e11_seconds"] = elapsed
        audit["available_at_start_seconds"] = available
        if available < 2.5 * 3600:
            raise TimeoutError(f"only {available / 60:.1f} minutes remain")

        # Point the shared pixel path at the E11 recipe. These are the same process globals
        # the parent notebook's readers consult, so the override has to happen before any
        # slot is picked or any pixel is decoded, and nothing after this point may assume
        # the parent arm's values.
        globals().update(
            SLOTS=list(E11_SLOTS),
            N_SLOT=len(E11_SLOTS),
            CACHE_SLICES=int(E11_CACHE_SLICES),
            IMG=int(E11_IMG),
            CACHE_IMG=int(E11_IMG),
            CROP_MM=float(E11_CROP_MM),
        )
        log(
            f"E11 recipe: {[s[0] for s in E11_SLOTS]} at {E11_CROP_MM:.0f} mm, "
            f"{E11_IMG} px, {E11_CACHE_SLICES} slices/slot"
        )

        train = pd.read_csv(ROOT / "train.csv", dtype={"StudyInstanceUID": str})
        train_series = pd.read_csv(
            ROOT / "train_series.csv",
            dtype={"StudyInstanceUID": str, "SeriesInstanceUID": str},
        )
        if len(train) != 4407:
            raise RuntimeError(f"unexpected train study count {len(train)}")
        plane = dict(zip(train_series.SeriesInstanceUID, train_series.Anatomical_Plane))
        headers = annotate(walk("train_series"))
        audit["availability"] = _v52_e11_availability(headers, plane)

        studies, pixels, slot_mask = build_cache(
            pick_slots(headers, plane), plane, lat_of(headers, "train-e11 "), "train-e11"
        )
        by_uid = {str(uid): i for i, uid in enumerate(studies)}
        missing = [uid for uid in train.StudyInstanceUID if uid not in by_uid]
        if missing:
            raise RuntimeError(f"{len(missing)} train studies absent from cache")
        order = np.array([by_uid[uid] for uid in train.StudyInstanceUID], dtype=np.int64)
        pixels, slot_mask = pixels[order], slot_mask[order]
        fill = float(slot_mask.mean())
        per_slot = {
            name: float(slot_mask[:, k].mean())
            for k, (name, _, _, _) in enumerate(E11_SLOTS)
        }
        audit["slot_fill"] = per_slot
        audit["overall_fill"] = fill
        for name, value in per_slot.items():
            log(f"E11 slot fill {name}: {value:.1%}")
        if fill < E11_MIN_FILL:
            raise RuntimeError(f"E11 slot fill {fill:.1%} below the {E11_MIN_FILL:.0%} floor")

        features, token_mask = encode_radimagenet(pixels, slot_mask, device)
        del pixels, slot_mask, headers
        gc.collect()

        y, weights, gold = make_targets(train)
        if int(gold.sum()) != 58:
            raise RuntimeError(f"expected 58 fully gold studies, observed {int(gold.sum())}")
        groups = report_groups(train)
        if len(np.unique(groups)) < 4000:
            raise RuntimeError("unexpected report-group collapse")

        splits = list(GroupKFold(5).split(features, groups=groups))
        fold_id = np.full(len(train), -1, dtype=np.int8)
        folds = []
        oof = np.zeros_like(y, dtype=np.float32)
        for fold, (tr, va) in enumerate(splits):
            if set(groups[tr]).intersection(groups[va]):
                raise RuntimeError(f"report leakage in fold {fold}")
            fold_id[va] = fold
            state, score = train_fold(
                features, token_mask, y, weights, tr, va, fold, device
            )
            if state is None:
                raise RuntimeError(f"fold {fold} produced no checkpoint")
            head = FoundationQueryHead().to(device)
            head.load_state_dict(state, strict=True)
            oof[va] = predict_head(head, features, token_mask, va, device)
            folds.append({"fold": fold, "weak_auc": float(score), "state_dict": state})
            del head
            torch.cuda.empty_cache()
        if (fold_id < 0).any() or not np.isfinite(oof).all():
            raise RuntimeError("incomplete E11 OOF")

        weak_auc = macro_auc(y, oof)
        gold_auc = macro_auc(y[gold], oof[gold])
        audit["weak_oof_auc"] = float(weak_auc)
        audit["gold_oof_auc"] = float(gold_auc)
        log(f"E11 OOF weak macro AUC {weak_auc:.5f}")
        log(f"E11 OOF gold macro AUC {gold_auc:.5f} on 58 studies")

        # The question E11 exists to answer is not whether this arm is strong on its own but
        # whether it says something the portfolio does not already know. Both halves are
        # measured against the same 58 rows and the same rank basis the deployed blend uses.
        base_npz = find_input_file("oof.npz")
        with np.load(base_npz, allow_pickle=False) as bundle:
            if bundle["targets"].astype(str).tolist() != TARGETS:
                raise RuntimeError("E11 E2 OOF target order drift")
            if not np.array_equal(
                bundle["ids"].astype(str), train.StudyInstanceUID.astype(str).to_numpy()
            ):
                raise RuntimeError("E11 E2 OOF study order drift")
            base_prediction = bundle["pred"].astype(np.float64)
        base = _v52_rank_columns(base_prediction[gold])
        new = _v52_rank_columns(oof[gold].astype(np.float64))
        gold_y = train.loc[gold, TARGETS].to_numpy(np.float64)
        reference = _v52_target_auc(gold_y, base)
        audit["e2_base_gold_macro"] = float(np.mean([reference[t] for t in TARGETS]))
        ladder = {}
        for alpha in (0.20, 0.35, 0.50):
            scores = _v52_target_auc(gold_y, (1.0 - alpha) * base + alpha * new)
            ladder[f"{alpha:.2f}"] = {
                "macro": float(np.mean([scores[t] for t in TARGETS])),
                "per_target_delta": {t: float(scores[t] - reference[t]) for t in TARGETS},
            }
            log(f"E11 blend alpha={alpha:.2f} gold macro {ladder[f'{alpha:.2f}']['macro']:.5f}")
        audit["blend_vs_e2"] = ladder

        try:
            parent_oof = pd.read_csv(
                find_input_file("v52_oof.csv"), dtype={"StudyInstanceUID": str}
            )
            aligned = train[["StudyInstanceUID"]].merge(
                parent_oof, on="StudyInstanceUID", how="left", validate="one_to_one"
            )
            parent = _v52_rank_columns(aligned[TARGETS].to_numpy(np.float64)[gold])
            audit["correlation_with_parent_arm"] = {
                t: float(np.corrcoef(parent[:, i], new[:, i])[0, 1])
                for i, t in enumerate(TARGETS)
            }
            log(
                "E11 mean rank correlation with the parent arm: "
                f"{np.mean(list(audit['correlation_with_parent_arm'].values())):.3f}"
            )
        except FileNotFoundError:
            audit["correlation_with_parent_arm"] = None

        torch.save(
            {
                "version": "e11-radimagenet-resnet50-diverse-1",
                "targets": TARGETS,
                "encoder_sha256": audit["encoder_sha256"],
                "slots": [list(slot) for slot in E11_SLOTS],
                "crop_mm": E11_CROP_MM,
                "img": E11_IMG,
                "slices_per_plane": E11_CACHE_SLICES,
                "feature": "global_average_pool",
                "folds": folds,
                "weak_oof_auc": float(weak_auc),
                "gold_oof_auc": float(gold_auc),
            },
            output / "v52_e11_heads.pt",
        )
        oof_frame = pd.DataFrame(oof, columns=TARGETS)
        oof_frame.insert(0, "StudyInstanceUID", train.StudyInstanceUID)
        oof_frame["fold"] = fold_id
        oof_frame["is_gold"] = gold.astype(np.uint8)
        oof_frame.to_csv(output / "v52_e11_oof.csv", index=False)
        audit["status"] = "E11_TRAINED_E2_PRESERVED"
        audit["heads_sha256"] = _v52_sha256(output / "v52_e11_heads.pt")
        audit["oof_sha256"] = _v52_sha256(output / "v52_e11_oof.csv")
    except Exception as error:
        audit["status"] = "ERROR_E2_PRESERVED"
        audit["error"] = f"{type(error).__name__}: {error}"
        audit["traceback"] = traceback.format_exc()
        log(f"E11 preserves E2: {audit['error']}")
    finally:
        if preserved.is_file():
            shutil.copy2(preserved, primary)
        audit["primary_sha256"] = _v52_sha256(primary) if primary.is_file() else None
        audit_path.write_text(json.dumps(audit, indent=2, sort_keys=True) + "\n")


if ARM_MODE == "e11":
    main_v52_e11()
else:
    try:
        find_input_file("v52_radimagenet_heads.pt")
    except FileNotFoundError:
        main_v52()
    else:
        try:
            find_input_file("e10_contract.json")
        except FileNotFoundError:
            main_v52_pinned_e9b()
        else:
            main_v52_e10()

